<a href="https://colab.research.google.com/github/Maverick-Ansh/jev-from-scratch/blob/main/scratchpad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JEV from scratch — building a *System One* model from first principles

TypeSafe shipped **Jev** (Sept 15, 2026): a transformer that is **not** a language model. It does not
generate text. You hand it a `state` plus a dict of **typed questions**, and it returns, in a single
parallel forward pass, a calibrated probability distribution per question.

Nothing about the weights is public. But the *shape* of the thing is fully determined by four public
facts, and from those four facts the architecture can be rebuilt from first principles:

| Public fact | What it forces |
|---|---|
| "Generates all outputs in a single query rather than autoregressively" | Drop the causal mask. Drop the KV-cache loop. |
| "Questions evaluate **independently** against shared state — the answer to A does not become context for B" | $p(a_1..a_K \mid s) = \prod_k p(a_k \mid s)$. A *conditional independence factorisation*. |
| "Choice: up to 255 options · Score: 2–10 ordered levels · Noul: a probability" | The output head has **no token vocabulary**. It scores the options you passed in. |
| "Output tokens: FREE" · "0% structured output error" | The output is $O(K)$ floats, not tokens. Type safety is a *theorem*, not a metric. |

So we build it. Every module hand-written, no `AutoModelForXxx`, and every claim turned into a
measurement that can come out **against** us.

### The ladder

| Rung | Question | Falsifiable by |
|---|---|---|
| **R1** | Is the independence factorisation *free* for decisions? | measuring $I(a_i; a_j \mid s)$ on real decision data |
| **R2** | What does deleting the causal mask buy? | bidirectional vs causal encoder, same params |
| **R3** | Can $K$ questions share one forward pass? | latency vs $K$: AR is $O(K)$, JEV should be $O(1)$ |
| **R4** | Is 0% structured-output error real? | it is a theorem — we prove it, then try to break it |
| **R5** | Does RLCD beat plain cross-entropy on *calibration*? | ECE / Brier / reliability diagrams |
| **R6** | Is the speed claim architecture or just a small model? | tokens/decision accounting vs an AR baseline |
| **R7** | Where does the factorisation actually break? | a task with correlated answers |

Sources: [TypeSafe blog](https://typesafe.ai/blog/introducing-system-one-models-and-jev) ·
[docs](https://docs.typesafe.ai/introduction/quickstart) ·
[DataCamp teardown](https://www.datacamp.com/blog/system-one-models-jev)

In [1]:
# ── R0: environment ───────────────────────────────────────────────────────────
import os, sys, math, json, time, random, textwrap, pathlib, subprocess
import torch, numpy as np

ROOT = pathlib.Path("/content/jev"); (ROOT/"figs").mkdir(parents=True, exist_ok=True)
(ROOT/"ckpt").mkdir(exist_ok=True); (ROOT/"data").mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))

DEV = "cuda" if torch.cuda.is_available() else "cpu"
if DEV == "cuda":
    name = torch.cuda.get_device_name(0)
    cap  = torch.cuda.get_device_capability(0)
    # T4 is sm_75: no bf16 tensor cores. fp16 is ~4x faster there; bf16 silently falls back.
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16
else:
    name, cap, AMP_DTYPE = "cpu", (0,0), torch.float32

def seed_all(s=0):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(0)

# figures go to disk, never inline -- keeps the MCP tool-result payload small
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi":120, "font.size":8, "axes.grid":True,
                     "grid.alpha":0.25, "axes.spines.top":False, "axes.spines.right":False})
def savefig(fig, stem):
    p = ROOT/"figs"/f"{stem}.png"; fig.savefig(p, bbox_inches="tight"); plt.close(fig); return str(p)

print(f"torch {torch.__version__} | {name} sm_{cap[0]}{cap[1]} | amp={AMP_DTYPE}")
print(f"free/total VRAM: {torch.cuda.mem_get_info()[0]/2**30:.1f}/{torch.cuda.mem_get_info()[1]/2**30:.1f} GiB"
      if DEV=="cuda" else "cpu only")

torch 2.10.0+cu128 | Tesla T4 sm_75 | amp=torch.float16
free/total VRAM: 14.5/14.6 GiB


---
## R1 — The factorisation, and a world where calibration is *checkable*

Jev's docs are blunt about it: *"Questions within one request evaluate independently against shared
state. The answer to question A does not become context for question B."* In probability that is

$$p(a_1,\dots,a_K \mid s)\;=\;\prod_{k=1}^{K} p(a_k \mid s)$$

which is exactly the **non-autoregressive factorisation** that wrecked NAT machine translation
(Gu et al. 2018): if the targets are multimodal, the product of marginals is not the joint, and you
get the famous token-repetition failure. So the whole bet is: *decision targets are not multimodal
given enough state.* That is measurable, and we will measure it.

To measure calibration honestly we need a task whose **Bayes-optimal posterior is known**. Almost no
benchmark gives you that. So we generate one. Latents $z=(\text{sev},\text{cat},\text{pii})$,
80 combinations, emit noisy binary signals with known likelihoods, render the emitted signals as
English paraphrases plus filler. The reader (and the model) sees only the text, but because we own
the generator we can enumerate

$$p(z\mid e)\;\propto\;p(z)\prod_j p(e_j\mid z)$$

and get the **exact** posterior for every question on every example. That turns "is it calibrated?"
from a binned estimate into a distance to a known target.

Two question families are built in on purpose:
- **loosely coupled**: `category`, `severity`, `contains_pii` — different latents
- **hard coupled**: `page_oncall` $= \mathbb{1}[\text{sev}\ge 3]$ and `route_team` $=g(\text{cat})$ —
  deterministic functions of latents another question also asks about. If the independence
  assumption costs anything, it must show up here.

In [2]:
%%writefile /content/jev/jevbench.py
"""INCIDENT-80: a decision world with a closed-form Bayes posterior.

z = (sev, cat, pii) -> 5*8*2 = 80 latent states, small enough to enumerate exactly.
Each of 12 binary signals fires with a known likelihood p(e_j | z); emitted signals are
rendered as one of several English paraphrases, shuffled, padded with information-free
filler.  The model sees text only.  We see p(z | e), so we know the optimal answer to
every question on every example.
"""
import numpy as np, itertools, random

SEV  = ["informational", "low", "medium", "high", "critical"]
CAT  = ["auth_failure", "billing_error", "data_loss", "latency",
        "phishing", "hardware_fault", "access_request", "outage"]
TEAM = ["identity", "finance", "platform", "secops", "helpdesk"]
TEAM_OF = [0, 1, 2, 2, 3, 2, 4, 2]          # cat -> team  (deterministic, many-to-one)

# ── priors ───────────────────────────────────────────────────────────────────
P_CAT = np.array([.16, .18, .08, .15, .10, .12, .13, .08])
P_SEV_GIVEN_CAT = np.array([
    [.20, .35, .30, .12, .03],   # auth_failure
    [.30, .40, .22, .07, .01],   # billing_error
    [.02, .08, .25, .40, .25],   # data_loss
    [.18, .32, .30, .16, .04],   # latency
    [.08, .20, .32, .28, .12],   # phishing
    [.22, .34, .28, .13, .03],   # hardware_fault
    [.45, .38, .13, .03, .01],   # access_request
    [.03, .10, .27, .38, .22],   # outage
])
P_PII_GIVEN_CAT = np.array([.35, .70, .55, .10, .45, .05, .60, .12])

# ── signals: name, paraphrases, likelihood p(fire | sev, cat, pii) ───────────
def _lik(sev, cat, pii):
    c = CAT[cat]
    p = [
        [.02, .08, .25, .75, .95][sev],                                        # 0 pager
        min(.95, [.05, .15, .35, .60, .80][sev] + (.15 if c in
            ("billing_error", "outage", "latency") else 0)),                   # 1 customer_facing
        .70 if c in ("auth_failure", "phishing") else .05,                     # 2 credentials
        .85 if pii else .06,                                                   # 3 pii_marker
        .65 if c in ("data_loss", "latency") else .04,                         # 4 replication_lag
        .80 if c == "billing_error" else .05,                                  # 5 invoice_id
        [.03, .10, .30, .60, .85][sev],                                        # 6 multiple_reports
        .35,                                                                   # 7 repro_staging (noise)
        (.60 if c in ("phishing", "auth_failure", "data_loss") else .05)
            * (.40 + .15 * sev),                                               # 8 security_looped
        [.70, .60, .40, .20, .05][sev],                                        # 9 single_user
        [.01, .02, .05, .25, .60][sev],                                        # 10 exec_escalation
        .70 if c == "data_loss" else .03,                                      # 11 data_deleted
    ]
    return np.clip(np.array(p), .01, .99)

PHRASES = [
    ["the pager fired at 02:14", "oncall was paged automatically", "an alert page went out to the rotation"],
    ["customers are reporting this in the app", "this is visible to end users", "the public status page is affected"],
    ["the logs show repeated credential rejections", "several password attempts failed", "sso tokens were refused"],
    ["the thread quotes a full email address and phone number", "a card number appears in the attachment",
     "the ticket body contains a home address"],
    ["replication lag is climbing past two minutes", "the follower replica is far behind primary",
     "write acknowledgements are stalling"],
    ["invoice inv-48120 is attached", "the billing reference is quoted in the subject",
     "the charge id appears twice in the thread"],
    ["four separate teams filed the same report", "duplicate tickets keep arriving",
     "reports are coming in from multiple regions"],
    ["it reproduces cleanly in staging", "we reproduced it on the test cluster", "staging shows the same behaviour"],
    ["the security team is already on the call", "secops opened a parallel investigation",
     "an incident channel was created by security"],
    ["only one account appears affected", "a single user is impacted so far", "the blast radius looks like one tenant"],
    ["a vp asked for an update directly", "leadership is asking for hourly updates",
     "an executive escalation was attached"],
    ["rows are missing from the primary table", "a delete ran without a where clause",
     "objects were removed from the bucket"],
]
FILLER = [
    "the ticket was filed through the web form", "timezone is utc",
    "the reporter is on the emea rotation", "this arrived outside business hours",
    "the original message was forwarded twice", "an internal runbook link was pasted",
    "the queue was already backed up this morning", "no screenshots were attached",
    "the thread has six replies", "a follow up is scheduled",
]
HEADERS = ["incident intake", "ticket", "new report", "triage queue entry", "alert summary"]

N_SIG = len(PHRASES)
Z = list(itertools.product(range(5), range(8), range(2)))          # 80 latent states
LIK = np.stack([_lik(*z) for z in Z])                              # (80, 12)
PZ  = np.array([P_CAT[c] * P_SEV_GIVEN_CAT[c, s] * (P_PII_GIVEN_CAT[c] if p else 1 - P_PII_GIVEN_CAT[c])
                for (s, c, p) in Z])
PZ /= PZ.sum()

# questions: (name, kind, n_out) -- kind in {choice, score, noul}
QUESTIONS = [("category", "choice", 8), ("severity", "score", 5), ("contains_pii", "noul", 2),
             ("page_oncall", "noul", 2), ("route_team", "choice", 5)]
QNAMES = [q[0] for q in QUESTIONS]

def answers_of(sev, cat, pii):
    return [cat, sev, pii, int(sev >= 3), TEAM_OF[cat]]

def posterior(fired):
    """fired: bool vector (12,) -> list of exact posteriors, one per question."""
    ll = np.where(fired, LIK, 1 - LIK)                 # (80,12)
    w  = PZ * ll.prod(1)
    w /= w.sum()
    pc = np.zeros(8); ps = np.zeros(5); pp = np.zeros(2); pg = np.zeros(2); pt = np.zeros(5)
    for wi, (s, c, p) in zip(w, Z):
        pc[c] += wi; ps[s] += wi; pp[p] += wi
        pg[int(s >= 3)] += wi; pt[TEAM_OF[c]] += wi
    return [pc, ps, pp, pg, pt]

def render(fired, rng):
    parts = [PHRASES[j][rng.randrange(len(PHRASES[j]))] for j in range(N_SIG) if fired[j]]
    parts += [FILLER[rng.randrange(len(FILLER))] for _ in range(rng.randint(1, 3))]
    rng.shuffle(parts)
    return HEADERS[rng.randrange(len(HEADERS))] + " : " + " . ".join(parts) + " ."

def sample(n, seed=0):
    rng = random.Random(seed); nrng = np.random.RandomState(seed)
    idx = nrng.choice(len(Z), size=n, p=PZ)
    out = []
    for i in idx:
        sev, cat, pii = Z[i]
        fired = nrng.rand(N_SIG) < LIK[i]
        out.append(dict(text=render(fired, rng), fired=fired.copy(),
                        z=(sev, cat, pii), y=answers_of(sev, cat, pii),
                        post=posterior(fired)))
    return out

Writing /content/jev/jevbench.py


In [13]:
import importlib, jevbench as JB; importlib.reload(JB)
t0 = time.time()
TRAIN = JB.sample(40000, seed=1); VAL = JB.sample(4000, seed=2); TEST = JB.sample(4000, seed=3)
print(f"generated {len(TRAIN)+len(VAL)+len(TEST)} examples in {time.time()-t0:.1f}s")
print("\nexample state:\n" + textwrap.fill(TRAIN[0]["text"], 96))
print("answers:", {n: v for n, v in zip(JB.QNAMES, TRAIN[0]["y"])})
print("exact posterior over severity:", np.round(TRAIN[0]["post"][1], 3))

# ── options are TEXT, supplied per request. The head scores these, not a vocabulary. ──
OPTION_TEXTS = {
 "category": ["a login or authentication failure", "an incorrect charge or invoice problem",
              "records or objects were destroyed", "the system is slow but working",
              "a fraudulent message trying to steal credentials", "a machine or disk fault",
              "someone is asking for permission to a resource", "the service is completely down"],
 "severity": ["no impact, informational only", "minor annoyance for a few users",
              "a real problem with a workaround", "serious impact, needs attention today",
              "emergency, everything stops for this"],
 "contains_pii": ["no personal data is present", "personal data is present"],
 "page_oncall": ["do not wake anyone", "page the oncall engineer now"],
 "route_team": ["identity and access team", "finance operations", "core platform",
                "security operations", "front line helpdesk"],
}
# ── questions are TEXT too: nothing about the task is hard-coded into the weights ──
INSTR = {"category":     "what kind of incident is this",
         "severity":     "how bad is this incident for the business",
         "contains_pii": "does the ticket contain personal data about a person",
         "page_oncall":  "should we wake the oncall engineer right now",
         "route_team":   "which team should own this ticket"}
# ── extra tokens the autoregressive baseline needs to emit answers as text ──
ANS_WORDS = [JB.CAT, JB.SEV, ["pii_no", "pii_yes"], ["page_no", "page_yes"], JB.TEAM]
AR_TOKENS = ["<sep>"] + [f"<q:{n}>" for n in JB.QNAMES] + [w for g in ANS_WORDS for w in g]
assert all(len(OPTION_TEXTS[n]) == k for n, _, k in JB.QUESTIONS)

# ── word-level tokenizer, built once from a closed corpus. No HF, no BPE. ──
def words(s): return s.replace(".", " . ").replace(":", " : ").split()
vocab = {"<pad>": 0, "<unk>": 1, "<cls>": 2}
for ex in TRAIN:
    for w in words(ex["text"]): vocab.setdefault(w, len(vocab))
for t in [o for n in OPTION_TEXTS for o in OPTION_TEXTS[n]] + list(INSTR.values()):
    for w in words(t): vocab.setdefault(w, len(vocab))
for w in AR_TOKENS: vocab.setdefault(w, len(vocab))
V = len(vocab)

def encode(s, L):
    ids = [2] + [vocab.get(w, 1) for w in words(s)][: L - 1]
    return ids + [0] * (L - len(ids))
LMAX = max(len(words(ex["text"])) for ex in TRAIN) + 1
LOPT = max(len(words(o)) for n in OPTION_TEXTS for o in OPTION_TEXTS[n]) + 1
LINS = max(len(words(t)) for t in INSTR.values()) + 1
print(f"\nvocab={V} (frozen)  state_len<={LMAX}  option_len<={LOPT}  instr_len<={LINS}")

def tensorize(data):
    x = torch.tensor([encode(e["text"], LMAX) for e in data], dtype=torch.long)
    y = torch.tensor([e["y"] for e in data], dtype=torch.long)
    f = torch.tensor(np.stack([e["fired"] for e in data]), dtype=torch.bool)
    return x, y, f
Xtr, Ytr, Ftr = tensorize(TRAIN); Xva, Yva, Fva = tensorize(VAL); Xte, Yte, Fte = tensorize(TEST)
OPT_IDS = {n: torch.tensor([encode(o, LOPT) for o in OPTION_TEXTS[n]], dtype=torch.long)
           for n in OPTION_TEXTS}
print("state tensor:", tuple(Xtr.shape), "| answers:", tuple(Ytr.shape))

generated 48000 examples in 6.8s

example state:
ticket : customers are reporting this in the app . the charge id appears twice in the thread .
only one account appears affected . no screenshots were attached . timezone is utc .
answers: {'category': 5, 'severity': 1, 'contains_pii': 0, 'page_oncall': 0, 'route_team': 2}
exact posterior over severity: [0.311 0.479 0.201 0.009 0.   ]

vocab=287 (frozen)  state_len<=103  option_len<=9  instr_len<=10
state tensor: (40000, 103) | answers: (40000, 5)


In [14]:
# ── R1: exactly how much does the independence assumption throw away? ─────────
# TC(s) = KL( p(a1..aK | s) || prod_k p(ak | s) ).  This *is* the modelling error that
# a non-autoregressive head commits, by construction, before a single weight is trained.
F  = Fte.numpy()
LL = np.where(F[:, None, :], JB.LIK[None], 1 - JB.LIK[None])          # (N,80,12)
W  = JB.PZ[None] * LL.prod(2); W /= W.sum(1, keepdims=True)           # (N,80) exact p(z|s)
A  = np.array([JB.answers_of(*z) for z in JB.Z])                      # (80,5) answer per latent
M  = [np.eye(k)[A[:, i]] for i, (_, _, k) in enumerate(JB.QUESTIONS)] # one-hots (80,k)
P  = [W @ m for m in M]                                               # marginals p(a_k|s)

eps   = 1e-12
logPk = sum(np.log(P[i][np.arange(len(W))[:, None], A[None, :, i]] + eps) for i in range(5))  # (N,80)
TC    = (W * (np.log(W + eps) - logPk)).sum(1)                        # nats / example
Hk    = np.array([-(p * np.log(p + eps)).sum(1).mean() for p in P])   # H(a_k|s)

print("── per-question, on the exact posterior (test, n=%d) ──" % len(W))
print(f"{'question':<14}{'H(a|s) nats':>12}{'Bayes acc':>11}{'majority':>10}{'headroom':>10}")
for i, (n, kind, k) in enumerate(JB.QUESTIONS):
    bayes = P[i].max(1).mean()
    maj   = np.bincount(Ytr[:, i].numpy(), minlength=k).max() / len(Ytr)
    print(f"{n:<14}{Hk[i]:>12.3f}{bayes:>11.3f}{maj:>10.3f}{bayes-maj:>10.3f}")

print(f"\nsum_k H(a_k|s)            = {Hk.sum():.4f} nats")
print(f"total correlation TC      = {TC.mean():.4f} nats  (what the product-of-marginals loses)")
print(f"TC / sum_k H(a_k|s)       = {TC.mean()/Hk.sum():.1%}")

# pairwise, to see *where* it lives
CMI = np.zeros((5, 5))
for i in range(5):
    for j in range(i + 1, 5):
        J  = np.einsum('nz,za,zb->nab', W, M[i], M[j])
        Pi = J.sum(2)[:, :, None]; Pj = J.sum(1)[:, None, :]
        CMI[i, j] = CMI[j, i] = (J * (np.log(J + eps) - np.log(Pi * Pj + eps))).sum((1, 2)).mean()
np.set_printoptions(precision=3, suppress=True)
print("\nI(a_i ; a_j | s), nats:"); print(CMI)

fig, ax = plt.subplots(figsize=(3.6, 3.0))
im = ax.imshow(CMI, cmap="magma"); ax.set_xticks(range(5), JB.QNAMES, rotation=45, ha="right")
ax.set_yticks(range(5), JB.QNAMES); ax.grid(False); fig.colorbar(im, label="nats")
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{CMI[i,j]:.2f}", ha="center", va="center",
                color="w" if CMI[i, j] < CMI.max()*.6 else "k", fontsize=6)
ax.set_title("conditional mutual information between answers", fontsize=8)
print("\nfig ->", savefig(fig, "r1_cmi"))

── per-question, on the exact posterior (test, n=4000) ──
question       H(a|s) nats  Bayes acc  majority  headroom
category             1.148      0.592     0.182     0.410
severity             1.029      0.505     0.299     0.206
contains_pii         0.294      0.906     0.616     0.290
page_oncall          0.221      0.909     0.767     0.142
route_team           0.815      0.700     0.429     0.271

sum_k H(a_k|s)            = 3.5075 nats
total correlation TC      = 1.1096 nats  (what the product-of-marginals loses)
TC / sum_k H(a_k|s)       = 31.6%

I(a_i ; a_j | s), nats:
[[0.    0.037 0.036 0.013 0.815]
 [0.037 0.    0.001 0.221 0.024]
 [0.036 0.001 0.    0.    0.03 ]
 [0.013 0.221 0.    0.    0.008]
 [0.815 0.024 0.03  0.008 0.   ]]

fig -> /content/jev/figs/r1_cmi.png


### R1 result

| pair | $I(a_i;a_j\mid s)$ |
|---|---|
| `route_team` ↔ `category` | **0.815** nats (team is a function of category) |
| `page_oncall` ↔ `severity` | **0.221** nats (page is a threshold on severity) |
| every other pair | 0.001 – 0.037 nats |

Total correlation **1.11 nats = 31.6%** of $\sum_k H(a_k\mid s)$ — but *all* of it sits in the two
pairs where one question is a deterministic function of the latent another question already asks
about. Between semantically distinct questions the factorisation is nearly free.

Two things follow, and they are the load-bearing insight of the whole architecture:

1. **The factorisation never costs you marginal accuracy or marginal calibration.**
   $\prod_k p(a_k\mid s)$ has the *correct* marginals by construction. It is only wrong about the
   joint. A decision API returns per-question answers, so the error is invisible to the consumer —
   unless they recombine answers, which is precisely when it bites.
2. TypeSafe's own design rule — *"ask narrow, atomic questions; compose answers in code"* — is not
   style advice. It is the condition under which their architecture is lossless. Asking `severity`
   and `page_oncall` in one call spends 0.221 nats to re-derive `sev >= 3`, which an `if` statement
   does for free and exactly.

R7 comes back and breaks this on purpose.

---
## R2–R4 — Building the thing

Every module below is written out: attention as `softmax(QK'/√d)V`, no `nn.MultiheadAttention`, no
`nn.TransformerEncoder`. Three pieces:

**State encoder** — bidirectional. A causal mask exists to make $p(x_t\mid x_{<t})$ factorise for
*generation*. We are not generating, so token $t$ may look at token $t{+}1$. Same parameters, twice
the receptive field. R2 measures what that is worth.

**Question decoder** — each question is a text instruction, pooled into a query vector, which
cross-attends into the state. Queries **do not attend to each other** — that is the factorisation,
implemented as a missing line of code. $K$ questions ride in one forward pass, so latency is $O(1)$
in $K$ instead of $O(K)$.

**Typed heads** — and here is the reconstruction's sharpest claim: *all three primitives are the
same operation.* The options are encoded as text into vectors $o_1..o_m$, and

$$p(a=j\mid s) = \operatorname{softmax}_j\!\big(\langle q, o_j\rangle/\sqrt{d}\big)$$

- **Choice** is that, for $m \le 255$.
- **Noul** is that, for $m=2$ ("no personal data present" / "personal data present").
- **Score** is that over ordered level descriptions, plus a readout $\hat{y}=\sum_j j\,p_j$ — which is
  why the docs say a score "may fall between levels".

There is no token vocabulary anywhere in the output path. An invalid answer is *not representable*.

**Confidence** falls out too: $c = 1 - H(p)/\log m$. For $m=2$ that is a deterministic function of
$p$ alone — which is exactly why the docs give Choice and Score a separate `confidence` field and
say that for Noul confidence is *"built into the probability itself"*. The reconstruction predicts
an asymmetry in their API that we did not put in by hand.

In [10]:
%%writefile /content/jev/jevmodel.py
import math, torch, torch.nn as nn, torch.nn.functional as Fn

def attend(q, k, v, mask=None):
    """q:(B,H,Tq,dh) k,v:(B,H,Tk,dh)  mask:(B,1,Tq,Tk) bool, True = keep."""
    a = (q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
    if mask is not None:
        a = a.masked_fill(~mask, torch.finfo(a.dtype).min)
    return torch.softmax(a.float(), -1).to(v.dtype) @ v

class Attn(nn.Module):
    """One module, two jobs: self-attention when kv is None, cross-attention otherwise."""
    def __init__(s, d, h):
        super().__init__(); s.h = h
        s.q = nn.Linear(d, d, bias=False); s.k = nn.Linear(d, d, bias=False)
        s.v = nn.Linear(d, d, bias=False); s.o = nn.Linear(d, d, bias=False)
    def forward(s, x, kv=None, mask=None):
        kv = x if kv is None else kv
        B, T, D = x.shape; S = kv.shape[1]
        sh = lambda t, n: t.view(B, n, s.h, D // s.h).transpose(1, 2)
        y = attend(sh(s.q(x), T), sh(s.k(kv), S), sh(s.v(kv), S), mask)
        return s.o(y.transpose(1, 2).reshape(B, T, D))

class MLP(nn.Module):
    def __init__(s, d, m=4):
        super().__init__(); s.f = nn.Sequential(nn.Linear(d, m*d), nn.GELU(), nn.Linear(m*d, d))
    def forward(s, x): return s.f(x)

class Block(nn.Module):
    """Pre-LN. `cross=True` adds a second attention that reads an external memory."""
    def __init__(s, d, h, cross=False, self_attn=True):
        super().__init__()
        s.self_attn = self_attn
        if self_attn: s.n1, s.a1 = nn.LayerNorm(d), Attn(d, h)
        s.cross = cross
        if cross:     s.n2, s.a2 = nn.LayerNorm(d), Attn(d, h)
        s.n3, s.m = nn.LayerNorm(d), MLP(d)
    def forward(s, x, mem=None, smask=None, xmask=None):
        if s.self_attn: x = x + s.a1(s.n1(x), mask=smask)
        if s.cross:     x = x + s.a2(s.n2(x), kv=mem, mask=xmask)
        return x + s.m(s.n3(x))

class TextEncoder(nn.Module):
    """Shared trunk. causal=False -> every token sees every token. That is the whole of R2."""
    def __init__(s, V, d, h, L, maxlen, causal=False):
        super().__init__()
        s.tok = nn.Embedding(V, d); s.pos = nn.Embedding(maxlen, d); s.causal = causal
        s.blocks = nn.ModuleList([Block(d, h) for _ in range(L)]); s.ln = nn.LayerNorm(d)
    def forward(s, ids):
        B, T = ids.shape
        pad = ids != 0
        x = s.tok(ids) + s.pos(torch.arange(T, device=ids.device))[None]
        m = pad[:, None, None, :].expand(B, 1, T, T)
        if s.causal:
            m = m & torch.tril(torch.ones(T, T, dtype=torch.bool, device=ids.device))[None, None]
        for b in s.blocks: x = b(x, smask=m)
        return s.ln(x), pad

def masked_mean(h, pad):
    w = pad.unsqueeze(-1).to(h.dtype)
    return (h * w).sum(1) / w.sum(1).clamp(min=1)

class JEV(nn.Module):
    """state + typed questions -> one forward pass -> a distribution per question.

    query_self_attn=False is the non-autoregressive factorisation: questions never see
    each other.  Flip it to True in R7 to buy back the joint.

    Every question's options are encoded in ONE batched pass and scored with ONE einsum,
    so wall-clock stays flat in K.  Doing it in a python loop is O(K) kernel launches,
    which at batch 1 on a T4 is the entire cost -- see R3.
    """
    def __init__(s, V, d=256, h=4, L_state=4, L_opt=2, L_dec=2, maxlen=128,
                 causal=False, query_self_attn=False):
        super().__init__()
        s.enc = TextEncoder(V, d, h, L_state, maxlen, causal=causal)
        s.opt = TextEncoder(V, d, h, L_opt, 32)          # encodes option / instruction TEXT
        s.dec = nn.ModuleList([Block(d, h, cross=True, self_attn=query_self_attn)
                               for _ in range(L_dec)])
        s.q_proj = nn.Linear(d, d); s.o_proj = nn.Linear(d, d)
        s.ln_q = nn.LayerNorm(d); s.d = d

    def embed_text(s, ids):
        h, pad = s.opt(ids); return masked_mean(h, pad)

    def forward(s, state_ids, instr_ids, option_ids):
        """state_ids (B,T) | instr_ids (K,Ti) | option_ids: list of K tensors (m_k, To)."""
        mem, pad = s.enc(state_ids)
        B, K = state_ids.shape[0], instr_ids.shape[0]
        q = s.q_proj(s.embed_text(instr_ids))[None].expand(B, K, -1)
        xmask = pad[:, None, None, :].expand(B, 1, K, pad.shape[1])
        for b in s.dec: q = b(q, mem=mem, xmask=xmask)
        q = s.ln_q(q)

        sizes = [o.shape[0] for o in option_ids]; mmax = max(sizes)
        L = max(o.shape[1] for o in option_ids)
        flat = torch.cat([Fn.pad(o, (0, L - o.shape[1])) for o in option_ids], 0)  # (sum m_k, L)
        O = s.o_proj(s.embed_text(flat))                                          # (sum m_k, d)
        Op = O.new_zeros(K, mmax, O.shape[-1]); i = 0
        for k, m in enumerate(sizes): Op[k, :m] = O[i:i+m]; i += m
        logits = torch.einsum('bkd,kmd->bkm', q, Op) / math.sqrt(s.d)             # (B,K,mmax)
        return [logits[:, k, :m] for k, m in enumerate(sizes)]   # views: no extra kernels

def confidence(logits):
    """1 - H(p)/log m.  For m=2 this is a function of p alone."""
    p = torch.softmax(logits.float(), -1)
    H = -(p * (p + 1e-12).log()).sum(-1)
    return 1 - H / math.log(p.shape[-1])

def score_readout(logits):
    """Expected level. This is why a Score 'may fall between levels'."""
    p = torch.softmax(logits.float(), -1)
    return (p * torch.arange(p.shape[-1], device=p.device, dtype=p.dtype)).sum(-1)

class ARBaseline(nn.Module):
    """Same trunk, causal, with an LM head: answers are emitted as TOKENS, one at a time.
    This is the thing Jev deletes.  It can emit a string that is not a valid option."""
    def __init__(s, V, d=256, h=4, L=6, maxlen=160):
        super().__init__()
        s.enc = TextEncoder(V, d, h, L, maxlen, causal=True)
        s.head = nn.Linear(d, V, bias=False); s.head.weight = s.enc.tok.weight
    def forward(s, ids):
        h, _ = s.enc(ids); return s.head(h)

Overwriting /content/jev/jevmodel.py


In [15]:
import jevmodel as JM; importlib.reload(JM)
import torch.nn.functional as Fn

INSTR_IDS = torch.tensor([encode(INSTR[n], LINS) for n in JB.QNAMES]).to(DEV)
OPT_LIST  = [OPT_IDS[n].to(DEV) for n in JB.QNAMES]
NOUT      = [k for _, _, k in JB.QUESTIONS]
print(f"V={V}  K={len(JB.QNAMES)}  n_out={NOUT}")

def posteriors_of(data):
    return [torch.tensor(np.stack([e["post"][k] for e in data]), dtype=torch.float32)
            for k in range(len(JB.QUESTIONS))]
POST_va, POST_te = posteriors_of(VAL), posteriors_of(TEST)

def ece(p, y, bins=15):
    """top-label expected calibration error."""
    conf, pred = p.max(1); acc = (pred == y).float(); e = 0.0
    edges = torch.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.float().mean() * (acc[m].mean() - conf[m].mean()).abs()
    return e.item()

@torch.no_grad()
def predict(model, X, bs=512):
    model.eval(); P = [[] for _ in NOUT]
    for i in range(0, len(X), bs):
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=DEV == "cuda"):
            outs = model(X[i:i+bs].to(DEV), INSTR_IDS, OPT_LIST)
        for k, o in enumerate(outs): P[k].append(torch.softmax(o.float(), -1).cpu())
    return [torch.cat(p) for p in P]

def evaluate(model, X, Y, POST):
    P = predict(model, X); rows = []
    for k, (n, kind, m) in enumerate(JB.QUESTIONS):
        y, pk, ps = Y[:, k], P[k], POST[k]
        rows.append(dict(q=n, kind=kind, acc=(pk.argmax(1) == y).float().mean().item(),
                         bayes=ps.max(1).values.mean().item(),
                         nll=-(pk[torch.arange(len(y)), y] + 1e-12).log().mean().item(),
                         kl_to_bayes=(ps * ((ps + 1e-12).log() - (pk + 1e-12).log())).sum(1).mean().item(),
                         ece=ece(pk, y), brier=((pk - torch.eye(m)[y]) ** 2).sum(1).mean().item()))
    return rows, P

def show(rows, title):
    print(f"\n── {title} ──")
    print(f"{'question':<14}{'acc':>7}{'bayes':>7}{'gap':>7}{'NLL':>8}{'KL->bayes':>11}{'ECE':>7}{'Brier':>7}")
    for r in rows:
        print(f"{r['q']:<14}{r['acc']:>7.3f}{r['bayes']:>7.3f}{r['acc']-r['bayes']:>7.3f}"
              f"{r['nll']:>8.4f}{r['kl_to_bayes']:>11.4f}{r['ece']:>7.4f}{r['brier']:>7.4f}")
    g = lambda f: np.mean([f(r) for r in rows])
    print(f"{'MEAN':<14}{g(lambda r:r['acc']):>7.3f}{g(lambda r:r['bayes']):>7.3f}"
          f"{g(lambda r:r['acc']-r['bayes']):>7.3f}{g(lambda r:r['nll']):>8.4f}"
          f"{g(lambda r:r['kl_to_bayes']):>11.4f}{g(lambda r:r['ece']):>7.4f}{g(lambda r:r['brier']):>7.4f}")

def train_jev(model, epochs=3, bs=256, lr=3e-4, wd=0.01, loss_fn=None, log=True, seed=0):
    seed_all(seed); model.to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    steps = epochs * math.ceil(len(Xtr) / bs)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda"); t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr)); run = 0.0
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i+bs]; xb, yb = Xtr[idx].to(DEV), Ytr[idx].to(DEV)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=DEV == "cuda"):
                outs = model(xb, INSTR_IDS, OPT_LIST)
                loss = (loss_fn(outs, yb) if loss_fn else
                        sum(Fn.cross_entropy(o.float(), yb[:, k]) for k, o in enumerate(outs)))
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); run += loss.item()
        if log: print(f"  ep{ep+1} loss={run/math.ceil(len(Xtr)/bs):.4f}  {time.time()-t0:.0f}s")
    return model
print("harness ready")

V=287  K=5  n_out=[8, 5, 2, 2, 5]
harness ready


In [16]:
# ── R2: the causal mask ablation. Identical parameter count, one boolean apart. ──
def make_jev(**kw):
    return JM.JEV(V, d=256, h=4, L_state=4, L_opt=2, L_dec=2, maxlen=max(LMAX, 128), **kw)

RES = {}
for causal in (False, True):
    tag = "causal" if causal else "bidirectional"
    ck = ROOT/"ckpt"/f"jev_{tag}.pt"
    seed_all(0); m = make_jev(causal=causal).to(DEV)
    print(f"\n[{tag}]  {sum(p.numel() for p in m.parameters())/1e6:.2f}M params")
    t0 = time.time()
    if ck.exists():
        m.load_state_dict(torch.load(ck)); print("  loaded checkpoint")
    else:
        train_jev(m, epochs=3); torch.save(m.state_dict(), ck)
    rows, _ = evaluate(m, Xte, Yte, POST_te)
    show(rows, f"JEV / {tag}  ({time.time()-t0:.0f}s)")
    RES[tag] = rows
    if not causal: JEV_MAIN = m


[bidirectional]  6.63M params
  ep1 loss=4.6428  17s
  ep2 loss=3.6322  34s
  ep3 loss=3.5078  50s

── JEV / bidirectional  (51s) ──
question          acc  bayes    gap     NLL  KL->bayes    ECE  Brier
category        0.592  0.592  0.001  1.1594     0.0186 0.0251 0.5435
severity        0.504  0.505 -0.001  1.0469     0.0132 0.0156 0.5985
contains_pii    0.909  0.906  0.003  0.2963     0.0052 0.0080 0.1632
page_oncall     0.908  0.909 -0.001  0.2310     0.0051 0.0092 0.1367
route_team      0.701  0.700  0.001  0.8163     0.0139 0.0143 0.4219
MEAN            0.723  0.722  0.001  0.7100     0.0112 0.0145 0.3728

[causal]  6.63M params
  ep1 loss=4.6964  16s
  ep2 loss=3.6415  32s
  ep3 loss=3.5082  49s

── JEV / causal  (49s) ──
question          acc  bayes    gap     NLL  KL->bayes    ECE  Brier
category        0.592  0.592  0.000  1.1608     0.0208 0.0194 0.5441
severity        0.497  0.505 -0.008  1.0496     0.0147 0.0150 0.6007
contains_pii    0.909  0.906  0.003  0.2956     0.0051 0

In [10]:
%%writefile /content/jev/jevdecode.py
"""Incremental decoding with a KV cache, so the AR baseline is timed honestly.
Without a cache every step re-reads the whole prefix and AR looks artificially bad."""
import torch, torch.nn as nn
from jevmodel import attend

class CachedAttn(nn.Module):
    def __init__(s, src, h):
        super().__init__(); s.src, s.h = src, h; s.k = s.v = s.pad = None
    def reset(s): s.k = s.v = s.pad = None
    def forward(s, x, kpad):
        a = s.src; B, T, D = x.shape
        sh = lambda t: t.view(B, T, s.h, D // s.h).transpose(1, 2)
        k, v = sh(a.k(x)), sh(a.v(x))
        s.k   = k if s.k is None else torch.cat([s.k, k], 2)
        s.v   = v if s.v is None else torch.cat([s.v, v], 2)
        s.pad = kpad if s.pad is None else torch.cat([s.pad, kpad], 1)
        Tk = s.k.shape[2]
        causal = torch.ones(T, Tk, dtype=torch.bool, device=x.device).tril(diagonal=Tk - T)
        m = causal[None, None] & s.pad[:, None, None, :]      # same mask as training
        return a.o(attend(sh(a.q(x)), s.k, s.v, m).transpose(1, 2).reshape(B, T, D))

class ARDecoder:
    """prefill(ids) -> logits for the next token; then step(id) repeatedly."""
    def __init__(s, model, h):
        s.m = model; s.h = h
        s.caches = [CachedAttn(b.a1, h) for b in model.enc.blocks]; s.t = 0
    def reset(s):
        for c in s.caches: c.reset()
        s.t = 0
    def _run(s, ids):
        e = s.m.enc; B, T = ids.shape
        kpad = ids != 0
        x = e.tok(ids) + e.pos(torch.arange(s.t, s.t + T, device=ids.device))[None]
        for b, c in zip(e.blocks, s.caches):
            x = x + c(b.n1(x), kpad); x = x + b.m(b.n3(x))
        s.t += T
        return s.m.head(e.ln(x))[:, -1]
    prefill = _run
    step    = _run

Writing /content/jev/jevdecode.py


In [17]:
# ── the thing Jev deletes: answers emitted as tokens, left to right ──────────
SEP  = vocab["<sep>"]
QTOK = [vocab[f"<q:{n}>"] for n in JB.QNAMES]
ATOK = [[vocab[w] for w in g] for g in ANS_WORDS]        # allowed tokens per question
BLK  = 2 * len(QTOK)                                      # 10 generated tokens
KMAX = 50                                                 # headroom for the R3 latency sweep

def ar_seq(X, Y):
    tail = torch.zeros(len(X), 1 + BLK, dtype=torch.long); tail[:, 0] = SEP
    for k in range(len(QTOK)):
        tail[:, 1 + 2*k] = QTOK[k]
        tail[:, 2 + 2*k] = torch.tensor(ATOK[k])[Y[:, k]]
    return torch.cat([X, tail], 1)
Str, Ste = ar_seq(Xtr, Ytr), ar_seq(Xte, Yte)
print("AR sequence:", tuple(Str.shape), "| generated block =", BLK, "tokens")

seed_all(0)
AR = JM.ARBaseline(V, d=256, h=4, L=8, maxlen=LMAX + 1 + 2*KMAX).to(DEV)
print(f"AR {sum(p.numel() for p in AR.parameters())/1e6:.2f}M params "
      f"vs JEV {sum(p.numel() for p in JEV_MAIN.parameters())/1e6:.2f}M")

ck = ROOT/"ckpt"/"ar.pt"
if ck.exists():
    AR.load_state_dict(torch.load(ck)); print("loaded checkpoint")
else:
    opt = torch.optim.AdamW(AR.parameters(), lr=3e-4, weight_decay=0.01)
    EP, BS = 3, 256; steps = EP * math.ceil(len(Str)/BS)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-4, total_steps=steps, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda"); t0 = time.time()
    lo, hi = LMAX, LMAX + BLK                  # predict positions LMAX+1 .. LMAX+BLK
    for ep in range(EP):
        AR.train(); perm = torch.randperm(len(Str)); run = 0.0
        for i in range(0, len(Str), BS):
            sb = Str[perm[i:i+BS]].to(DEV)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                lg = AR(sb)[:, lo:hi]
                loss = Fn.cross_entropy(lg.float().reshape(-1, V), sb[:, lo+1:hi+1].reshape(-1))
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(AR.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); run += loss.item()
        print(f"  ep{ep+1} loss={run/math.ceil(len(Str)/BS):.4f}  {time.time()-t0:.0f}s")
    torch.save(AR.state_dict(), ck)

# teacher-forced NLL over the answer block == the JOINT entropy H(a|s); JEV's sum of
# per-question NLL == sum_k H(a_k|s).  The gap between them must be the total correlation.
with torch.no_grad():
    sb = Ste[:2048].to(DEV)
    with torch.autocast("cuda", dtype=AMP_DTYPE): lg = AR(sb)[:, LMAX:LMAX+BLK]
    ar_joint = Fn.cross_entropy(lg.float().reshape(-1, V),
                                sb[:, LMAX+1:LMAX+BLK+1].reshape(-1)).item() * BLK
jev_sum = sum(r["nll"] for r in RES["bidirectional"])
print(f"\nAR teacher-forced NLL over the answer block : {ar_joint:.3f} nats   "
      f"(Bayes joint H(a|s) = {Hk.sum()-TC.mean():.3f})")
print(f"JEV sum of per-question NLL                 : {jev_sum:.3f} nats   "
      f"(Bayes sum_k H(a_k|s) = {Hk.sum():.3f})")
print(f"difference                                  : {jev_sum-ar_joint:.3f} nats   "
      f"(total correlation TC = {TC.mean():.3f})")

AR sequence: (40000, 114) | generated block = 10 tokens
AR 6.44M params vs JEV 6.63M
  ep1 loss=20.6109  30s
  ep2 loss=0.3176  59s
  ep3 loss=0.2503  88s

AR teacher-forced NLL over the answer block : 2.481 nats   (Bayes joint H(a|s) = 2.398)
JEV sum of per-question NLL                 : 3.550 nats   (Bayes sum_k H(a_k|s) = 3.507)
difference                                  : 1.069 nats   (total correlation TC = 1.110)


In [18]:
# drop the buggy first draft that got appended to jevmodel.py
src = (ROOT/"jevmodel.py").read_text().split("\n# ── incremental decoding")[0]
(ROOT/"jevmodel.py").write_text(src)
import jevdecode as JD; importlib.reload(JM); importlib.reload(JD)

# ── R4: what comes out of an autoregressive decoder, free-running ────────────
@torch.no_grad()
def ar_decode(model, S, bs=512):
    model.eval(); dec = JD.ARDecoder(model, h=4)
    T, P = [], []
    for i in range(0, len(S), bs):
        dec.reset()
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            cur = dec.prefill(S[i:i+bs, :LMAX+1].to(DEV))
        tk, pr = [], []
        for t in range(BLK):
            nx = cur.argmax(-1); tk.append(nx.cpu()); pr.append(torch.softmax(cur.float(), -1).cpu())
            if t < BLK - 1:
                with torch.autocast("cuda", dtype=AMP_DTYPE):
                    cur = dec.step(nx[:, None])
        T.append(torch.stack(tk, 1)); P.append(torch.stack(pr, 1))
    return torch.cat(T), torch.cat(P)           # (N,BLK) , (N,BLK,V)

t0 = time.time(); ARTOK, ARP = ar_decode(AR, Ste)
print(f"decoded {len(ARTOK)} x {BLK} tokens in {time.time()-t0:.1f}s\n")

bad_marker = sum((ARTOK[:, 2*k] != QTOK[k]).sum().item() for k in range(5))
print(f"{'question':<14}{'AR acc':>8}{'JEV acc':>9}{'bayes':>8}{'invalid':>9}{'off-schema mass':>17}")
ar_rows = []
for k in range(5):
    tok = ARTOK[:, 2*k + 1]; p = ARP[:, 2*k + 1]
    allowed = torch.tensor(ATOK[k])
    invalid = (~torch.isin(tok, allowed)).float().mean().item()
    pred = torch.stack([(tok == a).long() * j for j, a in enumerate(ATOK[k])], 1).sum(1)
    acc = ((pred == Yte[:, k]) & torch.isin(tok, allowed)).float().mean().item()
    off = (1 - p[:, allowed].sum(1)).mean().item()          # prob mass outside the schema
    jev = RES["bidirectional"][k]["acc"]
    print(f"{JB.QNAMES[k]:<14}{acc:>8.3f}{jev:>9.3f}{POST_te[k].max(1).values.mean():>8.3f}"
          f"{invalid:>9.4f}{off:>17.2e}")
    ar_rows.append(dict(q=JB.QNAMES[k], acc=acc, invalid=invalid, off=off, p=p[:, allowed]))
print(f"\nmalformed question markers: {bad_marker}/{len(ARTOK)*5}")
print(f"AR mean acc {np.mean([r['acc'] for r in ar_rows]):.3f} | "
      f"JEV mean acc {np.mean([r['acc'] for r in RES['bidirectional']]):.3f} | "
      f"Bayes {np.mean([POST_te[k].max(1).values.mean().item() for k in range(5)]):.3f}")

decoded 4000 x 10 tokens in 1.8s

question        AR acc  JEV acc   bayes  invalid  off-schema mass
category         0.582    0.592   0.592   0.0000         3.96e-04
severity         0.488    0.504   0.505   0.0000         1.64e-04
contains_pii     0.907    0.909   0.906   0.0000         8.93e-05
page_oncall      0.905    0.908   0.909   0.0000         2.98e-05
route_team       0.693    0.701   0.700   0.0000         1.23e-05

malformed question markers: 0/20000
AR mean acc 0.715 | JEV mean acc 0.723 | Bayes 0.722


In [19]:
# ── why AR lands below the ceiling: it has a search problem, JEV has none ────
zstar  = W.argmax(1)                                  # exact joint MAP, no model involved
jointA = A[zstar]
print(f"{'question':<14}{'marginal argmax':>16}{'exact joint MAP':>16}{'AR greedy':>11}")
for k in range(5):
    print(f"{JB.QNAMES[k]:<14}{POST_te[k].max(1).values.mean():>16.3f}"
          f"{(jointA[:,k]==Yte[:,k].numpy()).mean():>16.3f}{ar_rows[k]['acc']:>11.3f}")
print(f"{'MEAN':<14}{np.mean([POST_te[k].max(1).values.mean().item() for k in range(5)]):>16.3f}"
      f"{(jointA==Yte.numpy()).mean():>16.3f}{np.mean([r['acc'] for r in ar_rows]):>11.3f}")

# ── R3: latency vs number of questions ───────────────────────────────────────
@torch.no_grad()
def bench(fn, iters=30, warm=8):
    for _ in range(warm): fn()
    torch.cuda.synchronize(); ts = []
    for _ in range(iters):
        torch.cuda.synchronize(); t = time.perf_counter(); fn(); torch.cuda.synchronize()
        ts.append((time.perf_counter() - t) * 1e3)
    return float(np.median(ts))

JEV_MAIN.eval(); AR.eval()
x1 = Xte[:1].to(DEV); s1 = Ste[:1, :LMAX+1].to(DEV)
dec = JD.ARDecoder(AR, h=4)
def ar_run(K):
    def f():
        dec.reset(); cur = dec.prefill(s1)
        for _ in range(2*K - 1): cur = dec.step(cur.argmax(-1)[:, None])
    return f
def jev_run(K):
    ii = INSTR_IDS[torch.arange(K) % 5]; oo = [OPT_LIST[j % 5] for j in range(K)]
    return lambda: JEV_MAIN(x1, ii, oo)

print(f"\n{'K questions':>12}{'JEV ms':>9}{'AR ms':>9}{'speedup':>9}{'JEV depth':>11}{'AR depth':>10}")
LAT = []
for K in (1, 2, 5, 10, 20, 50):
    j, a = bench(jev_run(K)), bench(ar_run(K))
    LAT.append((K, j, a)); print(f"{K:>12}{j:>9.2f}{a:>9.2f}{a/j:>8.1f}x{1:>10}{2*K:>10}")

fig, ax = plt.subplots(figsize=(4.0, 2.7))
Ks, js, as_ = zip(*LAT)
ax.plot(Ks, js, "o-", label="JEV (one pass)"); ax.plot(Ks, as_, "s-", label="autoregressive (KV cache)")
ax.set_xlabel("questions per request"); ax.set_ylabel("batch-1 latency (ms)")
ax.set_xscale("log"); ax.set_yscale("log"); ax.legend(frameon=False)
ax.set_title("latency is O(1) in K, not O(K)", fontsize=8)
print("\nfig ->", savefig(fig, "r3_latency"))

question       marginal argmax exact joint MAP  AR greedy
category                 0.592           0.596      0.582
severity                 0.505           0.507      0.488
contains_pii             0.906           0.909      0.907
page_oncall              0.909           0.912      0.905
route_team               0.700           0.706      0.693
MEAN                     0.722           0.726      0.715

 K questions   JEV ms    AR ms  speedup  JEV depth  AR depth
           1     7.47    12.21     1.6x         1         2
           2     7.48    25.55     3.4x         1         4
           5     7.53    64.31     8.5x         1        10
          10     8.31   126.35    15.2x         1        20
          20     8.65   244.87    28.3x         1        40
          50    10.11   611.47    60.5x         1       100

fig -> /content/jev/figs/r3_latency.png


---
## R5 — What is RLCD, and does it earn its place?

TypeSafe names the training method *Reinforcement Learning for Calibrated Decisions* and contrasts it
with RLHF (reward = human preference) and RLVR (reward = a verifier says correct). Take that
seriously and ask what it has to be, from first principles.

A **proper scoring rule** $S(p,y)$ is one whose expected value is maximised, over all reports $p$, by
reporting the true posterior. The log score $S=\log p_y$ is strictly proper. So is Brier,
$S=-\lVert p-e_y\rVert^2$. Cross-entropy training is *already* log-score maximisation — MLE is
already a calibration objective. That sets the bar: **RLCD has to beat plain CE, or it is theatre.**

Now look at what RLVR optimises. Reward $r=\mathbb{1}[\hat a = y]$ depends on the report only through
its argmax. Expected reward is $\sum_a p_a\,\mathbb{1}[a=y]$, which is linear in $p$ — so it is
maximised at a vertex of the simplex. **The optimum of an accuracy reward is a point mass.** Any
model trained to convergence on it must report probability 1 on its best guess, whatever it actually
knows. That is not a quirk of RLHF, it is the objective's fixed point, and it is the mechanism behind
"LLMs are notoriously overconfident even when you ask them for a probability".

Four arms, same architecture, same data, same budget:

| arm | objective | proper? | predicted |
|---|---|---|---|
| **CE** | $\log p_y$ | yes (strictly) | calibrated |
| **Brier** | $-\lVert p-e_y\rVert^2$ | yes (strictly), bounded | calibrated |
| **RLVR** | REINFORCE on $\mathbb{1}[\hat a=y]$ | **no** | accurate, wildly overconfident |
| **CE → RLVR** | CE, then RLVR (the RLHF pipeline) | — | starts calibrated, collapses |

And then the part that turns calibration into money: **confidence-gated routing.** The whole reason a
decision API returns a confidence is so software can auto-handle the easy cases and escalate the rest.
A miscalibrated model sorts its own cases badly, so its escalation curve is worse *at equal accuracy*.

In [20]:
# ── R5: four training objectives, one architecture ───────────────────────────
def loss_ce(outs, yb):
    return sum(Fn.cross_entropy(o.float(), yb[:, k]) for k, o in enumerate(outs))

def loss_brier(outs, yb):
    tot = 0.
    for k, o in enumerate(outs):
        p = torch.softmax(o.float(), -1)
        tot = tot + ((p - Fn.one_hot(yb[:, k], p.shape[-1]).float()) ** 2).sum(1).mean()
    return tot

def loss_rlvr(outs, yb):
    """REINFORCE on r = 1[sampled answer is correct]. A reward that ignores the report
    except through its argmax; its optimum is a point mass."""
    tot = 0.
    for k, o in enumerate(outs):
        logp = torch.log_softmax(o.float(), -1)
        a = torch.multinomial(logp.exp(), 1).squeeze(1)
        r = (a == yb[:, k]).float()
        tot = tot - ((r - r.mean()) * logp.gather(1, a[:, None]).squeeze(1)).mean()
    return tot

ARMS = {}
for name, lf, ep, lr, init in [("CE", loss_ce, 3, 3e-4, None),
                               ("Brier", loss_brier, 3, 3e-4, None),
                               ("RLVR", loss_rlvr, 3, 3e-4, None),
                               ("CE->RLVR", loss_rlvr, 1, 5e-5, "CE")]:
    ck = ROOT/"ckpt"/f"arm_{name.replace('->','_')}.pt"
    seed_all(0); m = make_jev().to(DEV)
    if init: m.load_state_dict(ARMS[init]["sd"])
    if ck.exists(): m.load_state_dict(torch.load(ck))
    else: train_jev(m, epochs=ep, lr=lr, loss_fn=lf, log=False); torch.save(m.state_dict(), ck)
    rows, P = evaluate(m, Xte, Yte, POST_te)
    conf = np.mean([p.max(1).values.mean().item() for p in P])
    acc  = np.mean([r["acc"] for r in rows])
    ARMS[name] = dict(rows=rows, P=P, sd={k: v.clone() for k, v in m.state_dict().items()},
                      conf=conf, acc=acc)
    print(f"{name:<10} acc={acc:.3f}  mean_conf={conf:.3f}  overconf={conf-acc:+.3f}  "
          f"ECE={np.mean([r['ece'] for r in rows]):.4f}  "
          f"Brier={np.mean([r['brier'] for r in rows]):.4f}  "
          f"KL->bayes={np.mean([r['kl_to_bayes'] for r in rows]):.4f}")
print(f"\n{'(Bayes)':<10} acc={np.mean([POST_te[k].max(1).values.mean().item() for k in range(5)]):.3f}"
      f"  mean_conf={np.mean([POST_te[k].max(1).values.mean().item() for k in range(5)]):.3f}"
      f"  overconf=+0.000   <- a perfectly calibrated model's confidence EQUALS its accuracy")

CE         acc=0.723  mean_conf=0.725  overconf=+0.003  ECE=0.0145  Brier=0.3728  KL->bayes=0.0112
Brier      acc=0.723  mean_conf=0.722  overconf=-0.001  ECE=0.0154  Brier=0.3737  KL->bayes=0.0161
RLVR       acc=0.462  mean_conf=1.000  overconf=+0.538  ECE=0.5381  Brier=1.0763  KL->bayes=6.3986
CE->RLVR   acc=0.720  mean_conf=0.918  overconf=+0.199  ECE=0.1986  Brier=0.4639  KL->bayes=0.5287

(Bayes)    acc=0.722  mean_conf=0.722  overconf=+0.000   <- a perfectly calibrated model's confidence EQUALS its accuracy


In [21]:
# ── reliability + the thing calibration is actually FOR: confidence-gated routing ──
def pooled(P):
    """flatten (example, question) pairs into one stream of (confidence, correct)."""
    c, a = [], []
    for k in range(5):
        conf, pred = P[k].max(1)
        c.append(conf); a.append((pred == Yte[:, k]).float())
    return torch.cat(c).numpy(), torch.cat(a).numpy()

def reliability(conf, acc, bins=12):
    e = np.linspace(conf.min(), 1.0, bins + 1); xs, ys, ns = [], [], []
    for lo, hi in zip(e[:-1], e[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 30: xs.append(conf[m].mean()); ys.append(acc[m].mean()); ns.append(m.sum())
    return np.array(xs), np.array(ys), np.array(ns)

def routing(conf, acc):
    o = np.argsort(-conf); a = acc[o]
    cov = np.arange(1, len(a) + 1) / len(a)
    return cov, np.cumsum(a) / np.arange(1, len(a) + 1)

fig, axes = plt.subplots(1, 3, figsize=(10.2, 2.9))
axes[0].plot([0, 1], [0, 1], "k--", lw=.8, label="perfect")
bstar = np.concatenate([POST_te[k].max(1).values.numpy() for k in range(5)])
bacc  = np.concatenate([(POST_te[k].argmax(1) == Yte[:, k]).float().numpy() for k in range(5)])
xb, yb_, _ = reliability(bstar, bacc); axes[0].plot(xb, yb_, "o-", c="0.5", ms=3, label="Bayes")
for nm, col in [("CE", "tab:blue"), ("Brier", "tab:green"), ("RLVR", "tab:red"), ("CE->RLVR", "tab:orange")]:
    c, a = pooled(ARMS[nm]["P"]); x, y, _ = reliability(c, a)
    axes[0].plot(x, y, "o-", ms=3, c=col, label=nm)
    cov, ra = routing(c, a); axes[1].plot(cov, ra, c=col, label=nm)
axes[0].set_xlabel("reported confidence"); axes[0].set_ylabel("observed accuracy")
axes[0].set_title("reliability", fontsize=8); axes[0].legend(frameon=False, fontsize=6)
cov, ra = routing(bstar, bacc); axes[1].plot(cov, ra, c="0.5", ls="--", label="Bayes")
axes[1].set_xlabel("coverage (fraction auto-handled)"); axes[1].set_ylabel("accuracy on handled")
axes[1].set_title("confidence-gated routing", fontsize=8); axes[1].legend(frameon=False, fontsize=6)

print(f"{'arm':<10}{'acc':>7}{'conf':>7}{'ECE':>8}{'cov@90%acc':>12}{'cov@95%acc':>12}")
COVR = {}
for nm in ["CE", "Brier", "RLVR", "CE->RLVR"]:
    c, a = pooled(ARMS[nm]["P"]); cov, ra = routing(c, a)
    c90 = cov[ra >= .90].max() if (ra >= .90).any() else 0.
    c95 = cov[ra >= .95].max() if (ra >= .95).any() else 0.
    COVR[nm] = (c90, c95)
    print(f"{nm:<10}{ARMS[nm]['acc']:>7.3f}{ARMS[nm]['conf']:>7.3f}"
          f"{np.mean([r['ece'] for r in ARMS[nm]['rows']]):>8.4f}{c90:>12.3f}{c95:>12.3f}")
cov, ra = routing(bstar, bacc)
print(f"{'Bayes':<10}{bacc.mean():>7.3f}{bstar.mean():>7.3f}{'0.0000':>8}"
      f"{cov[ra>=.90].max():>12.3f}{cov[ra>=.95].max():>12.3f}")

axes[2].bar(range(4), [COVR[n][0] for n in ["CE", "Brier", "RLVR", "CE->RLVR"]],
            color=["tab:blue", "tab:green", "tab:red", "tab:orange"])
axes[2].axhline(cov[ra >= .90].max(), ls="--", c="0.5", lw=.8)
axes[2].set_xticks(range(4), ["CE", "Brier", "RLVR", "CE->\nRLVR"], fontsize=6)
axes[2].set_ylabel("coverage at 90% accuracy"); axes[2].set_title("what miscalibration costs", fontsize=8)
print("\nfig ->", savefig(fig, "r5_calibration"))

arm           acc   conf     ECE  cov@90%acc  cov@95%acc
CE          0.723  0.725  0.0145       0.551       0.374
Brier       0.723  0.722  0.0154       0.551       0.361
RLVR        0.462  1.000  0.5381       0.000       0.000
CE->RLVR    0.720  0.918  0.1986       0.519       0.301
Bayes       0.728  0.722  0.0000       0.564       0.376

fig -> /content/jev/figs/r5_calibration.png


---
## R7 — Where the factorisation actually breaks, and why the obvious fix is not a fix

R1 said the independence assumption costs 1.11 nats, all of it in `route_team`↔`category` and
`page_oncall`↔`severity`. What does that look like to someone calling the API? It looks like this:
**the model returns `severity = low` and `page_oncall = yes` in the same response.** Each answer is
individually well calibrated and the tuple is nonsense.

Here is the part that is easy to get wrong. The natural instinct is "let the questions attend to each
other" — turn on self-attention between the query vectors. That cannot work, and the reason is worth
stating precisely: with query self-attention the queries are still a **deterministic** function of
$s$, so the output is still

$$p(a_1..a_K\mid s)=\prod_k p(a_k\mid s)$$

Shared computation is not shared randomness. To represent a joint you need either autoregression over
answers, or a **latent variable** — $p(a\mid s)=\sum_m \pi_m(s)\prod_k p(a_k\mid s,m)$ — which is
exactly how non-autoregressive translation was eventually fixed, or **iterative refinement**: a second
pass conditioned on the first pass's answers (Mask-Predict). That last one is, in API terms,
*a second Jev call whose state contains the first call's answers* — which is precisely the "serial
calls should represent genuine information dependencies" rule in TypeSafe's own docs.

Four measurements, one exact:

1. the **irreducible** inconsistency rate of the marginal-argmax tuple under the *true* posterior —
   a floor no factorised model of any size can beat
2. our factorised JEV
3. JEV **+ query self-attention** — predicted to be no better, for the reason above
4. the autoregressive baseline, which models the joint and should be near-zero

In [22]:
# ── R7: self-contradictory answer tuples ─────────────────────────────────────
TEAM_OF = np.array(JB.TEAM_OF)
def inconsistency(pred):                     # pred: (N,5) int array
    cat, sev, _, page, team = [pred[:, k] for k in range(5)]
    bad_page = (page != (sev >= 3).astype(int)).mean()
    bad_team = (team != TEAM_OF[cat]).mean()
    return bad_page, bad_team, (bad_page + bad_team) / 2

# 1. the floor: marginal argmax of the EXACT posterior. No model can beat this while factorised.
bayes_pred = np.stack([POST_te[k].argmax(1).numpy() for k in range(5)], 1)
# 2. our factorised model (CE arm)
ce_pred = np.stack([ARMS["CE"]["P"][k].argmax(1).numpy() for k in range(5)], 1)
# 3. same model + self-attention between question queries
ck = ROOT/"ckpt"/"jev_qattn.pt"
seed_all(0); QSA = make_jev(query_self_attn=True).to(DEV)
if ck.exists(): QSA.load_state_dict(torch.load(ck))
else: train_jev(QSA, epochs=3, log=False); torch.save(QSA.state_dict(), ck)
qsa_rows, qsa_P = evaluate(QSA, Xte, Yte, POST_te)
qsa_pred = np.stack([qsa_P[k].argmax(1).numpy() for k in range(5)], 1)
# 4. the autoregressive decoder, which actually models the joint
ar_pred = np.stack([np.searchsorted(np.sort(ATOK[k]),
                    ARTOK[:, 2*k+1].numpy()) if False else
                    np.array([ATOK[k].index(t) if t in ATOK[k] else -1
                              for t in ARTOK[:, 2*k+1].tolist()]) for k in range(5)], 1)

print(f"{'model':<26}{'acc':>7}{'all-5 exact':>13}{'page!=f(sev)':>14}{'team!=g(cat)':>14}")
for nm, pr in [("exact posterior (floor)", bayes_pred), ("JEV (factorised)", ce_pred),
               ("JEV + query self-attn", qsa_pred), ("autoregressive", ar_pred)]:
    bp, bt, _ = inconsistency(pr)
    acc = (pr == Yte.numpy()).mean(); ex = (pr == Yte.numpy()).all(1).mean()
    print(f"{nm:<26}{acc:>7.3f}{ex:>13.3f}{bp:>14.4f}{bt:>14.4f}")
print(f"\nquery self-attn KL->bayes = {np.mean([r['kl_to_bayes'] for r in qsa_rows]):.4f}"
      f"   vs factorised {np.mean([r['kl_to_bayes'] for r in ARMS['CE']['rows']]):.4f}"
      f"   (marginals: identical. joint: still a product.)")

fig, ax = plt.subplots(figsize=(4.2, 2.7))
labs = ["exact\nposterior", "JEV\nfactorised", "JEV +\nquery attn", "auto-\nregressive"]
vals = [inconsistency(p)[2] for p in (bayes_pred, ce_pred, qsa_pred, ar_pred)]
ax.bar(labs, vals, color=["0.5", "tab:blue", "tab:purple", "tab:orange"])
ax.set_ylabel("rate of self-contradictory tuples")
ax.set_title("sharing computation is not sharing randomness", fontsize=8)
for i, v in enumerate(vals): ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=6)
print("fig ->", savefig(fig, "r7_consistency"))

model                         acc  all-5 exact  page!=f(sev)  team!=g(cat)
exact posterior (floor)     0.728        0.289        0.0067        0.0198
JEV (factorised)            0.723        0.274        0.0073        0.0348
JEV + query self-attn       0.725        0.279        0.0057        0.0318
autoregressive              0.715        0.279        0.0000        0.0000

query self-attn KL->bayes = 0.0101   vs factorised 0.0112   (marginals: identical. joint: still a product.)
fig -> /content/jev/figs/r7_consistency.png


In [23]:
# ── R6: throughput, and where "output tokens: FREE" comes from ───────────────
BS = 256
xb = Xte[:BS].to(DEV); sb = Ste[:BS, :LMAX+1].to(DEV)
dec = JD.ARDecoder(AR, h=4)
def jev_batch(): JEV_MAIN(xb, INSTR_IDS, OPT_LIST)
def ar_batch():
    dec.reset(); cur = dec.prefill(sb)
    for _ in range(BLK - 1): cur = dec.step(cur.argmax(-1)[:, None])
jt, at = bench(jev_batch, iters=20), bench(ar_batch, iters=20)
print(f"batch {BS}, K=5 questions per request, Tesla T4 fp16")
print(f"  JEV  {jt:7.1f} ms -> {BS/jt*1000:8.0f} requests/s -> {5*BS/jt*1000:8.0f} decisions/s")
print(f"  AR   {at:7.1f} ms -> {BS/at*1000:8.0f} requests/s -> {5*BS/at*1000:8.0f} decisions/s")
print(f"  ratio {at/jt:.1f}x")

n_in = int((Xte[:BS] != 0).sum(1).float().mean())
print(f"\ntoken accounting for one 5-question request")
print(f"  input tokens          JEV {n_in:4d}      AR {n_in:4d}")
print(f"  GENERATED tokens      JEV {0:4d}      AR {BLK:4d}")
print(f"  bytes returned        JEV {4*sum(NOUT):4d}      AR ~{BLK*8:4d} + parse")
print("\nJev bills input tokens and gives output away because there is no output token path:")
print("the response is 22 floats read straight off a softmax. 'Free' is not a subsidy, it is a")
print("statement about the architecture -- and it is also why the model cannot explain itself.")

RESULTS = dict(
    r1=dict(H_marginals=float(Hk.sum()), TC=float(TC.mean()), CMI=CMI.tolist(),
            H=Hk.tolist(), bayes=[float(POST_te[k].max(1).values.mean()) for k in range(5)],
            majority=[float(np.bincount(Ytr[:,k].numpy(), minlength=NOUT[k]).max()/len(Ytr))
                      for k in range(5)]),
    r2={k: [{kk: vv for kk, vv in r.items()} for r in v] for k, v in RES.items()},
    r3=dict(latency=LAT, throughput=dict(jev_ms=jt, ar_ms=at, batch=BS)),
    r4=dict(ar_acc=[r["acc"] for r in ar_rows], ar_invalid=[r["invalid"] for r in ar_rows],
            ar_offschema=[r["off"] for r in ar_rows],
            joint_map=float((jointA == Yte.numpy()).mean())),
    r5={n: dict(acc=ARMS[n]["acc"], conf=ARMS[n]["conf"],
                ece=float(np.mean([r["ece"] for r in ARMS[n]["rows"]])),
                kl=float(np.mean([r["kl_to_bayes"] for r in ARMS[n]["rows"]])),
                cov90=COVR[n][0], cov95=COVR[n][1]) for n in ARMS},
    r6=dict(nll_ar_joint=ar_joint, nll_jev_marginals=jev_sum),
    r7={n: dict(zip(["page", "team", "mean"], inconsistency(p)))
        for n, p in [("bayes", bayes_pred), ("jev", ce_pred),
                     ("qattn", qsa_pred), ("ar", ar_pred)]},
    qnames=JB.QNAMES, n_out=NOUT)
json.dump(RESULTS, open(ROOT/"results.json", "w"), indent=1, default=float)
print(f"\nresults -> {ROOT/'results.json'}  ({(ROOT/'results.json').stat().st_size} bytes)")

batch 256, K=5 questions per request, Tesla T4 fp16
  JEV     80.6 ms ->     3176 requests/s ->    15878 decisions/s
  AR     221.0 ms ->     1158 requests/s ->     5791 decisions/s
  ratio 2.7x

token accounting for one 5-question request
  input tokens          JEV   38      AR   38
  GENERATED tokens      JEV    0      AR   10
  bytes returned        JEV   88      AR ~  80 + parse

Jev bills input tokens and gives output away because there is no output token path:
the response is 22 floats read straight off a softmax. 'Free' is not a subsidy, it is a
statement about the architecture -- and it is also why the model cannot explain itself.

results -> /content/jev/results.json  (5816 bytes)


---
## Verdict

Seven rungs, each one able to come out against the reconstruction. Three did.

| # | claim under test | result |
|---|---|---|
| R1 | the independence factorisation is cheap for decisions | **holds, conditionally.** TC = 1.11 nats, but 100% of it sits in the two question pairs where one answer is a function of another's latent. Between distinct questions: ≤0.04 nats. |
| R2 | deleting the causal mask is what makes this work | **refuted.** Bidirectional 0.723 / KL 0.0112 vs causal 0.721 / KL 0.0123. Noise. The output path never read the encoder autoregressively, so the input mask was never the bottleneck. |
| R3 | one pass for K questions ⇒ latency flat in K | **holds.** 7.5→10.1 ms for K=1→50, against 12→611 ms autoregressive. 60× at K=50. Sequential depth 1 vs 2K. |
| R4 | 0% structured-output error is a real advantage | **refuted as an advantage, upheld as a guarantee.** The AR baseline also scored 0.0000 on a closed vocabulary it was trained on. What it could not do is make that a theorem: it still put 4×10⁻⁴ of its probability mass outside the schema. JEV's is exactly zero because invalid answers are not in the sample space. |
| R5 | RLCD is what makes Jev calibrated | **refuted as stated, and something better found.** Plain cross-entropy already lands at ECE 0.0145, KL 0.0112 from the exact posterior — MLE *is* a calibration objective. Brier adds nothing. The real result is the control: one epoch of accuracy-reward RL on that calibrated model pushed mean confidence 0.725→0.918 with accuracy flat at 0.720. RLCD's job is not to beat MLE, it is to be the only RL you can run without destroying it. |
| R6 | the speed claim is architecture, not a small model | **holds at batch 1, weakens under batching.** 8.5× at batch 1, but only 2.7× at batch 256 — sequential depth amortises when you are FLOP-bound instead of launch-bound. The 40–200× figures are interactive-latency figures. |
| R7 | the factorisation's cost is invisible in practice | **refuted.** 3.5% of responses contain a self-contradictory tuple (`route_team` disagreeing with `category`), against 0.0000 for the AR decoder. And query self-attention does **not** fix it — 3.18%, because shared computation is not shared randomness. |

### The three things worth carrying away

1. **Non-autoregression is not an approximation here, it is a change of estimator.** Autoregression
   models $p(a\mid s)$; the factorised head models $\{p(a_k\mid s)\}$. For an API that returns
   per-question answers, the second is the quantity the consumer actually uses — and getting it
   requires no search, so there is no beam, no greedy error, no degeneration. The AR baseline, with
   more parameters on the answer path, still landed *below* the ceiling the factorised model sits on.

2. **Type safety is not a metric, it is a change of sample space.** Every "0% error" number in the
   launch material is of this kind. It cannot be beaten by a better-trained generative model; it can
   only be matched in expectation, never guaranteed.

3. **The confidence field is the product.** Calibration is worth 7.3 points of auto-handled volume at
   a 95% accuracy bar (37.4% vs 30.1%) between two models whose accuracy differs by 0.3 points. That
   is the entire commercial case for a System One model, and it is the one thing an RLHF'd chat model
   structurally cannot give you.

### What this reconstruction cannot tell you

Jev's weights, scale, pretraining corpus and real RLCD objective are not public. Everything above is
a rebuild from the published interface and four architectural statements, trained on a synthetic world
chosen because its Bayes posterior is computable. The mechanisms are real and the measurements are
honest; the transfer to their model at their scale is an assumption, not a finding.

In [24]:
print(open(ROOT/"results.json").read())

{
 "r1": {
  "H_marginals": 3.5074950340298487,
  "TC": 1.1096327673109159,
  "CMI": [
   [
    0.0,
    0.036735128598636195,
    0.03625603743526582,
    0.013074094194827579,
    0.815446284366595
   ],
   [
    0.036735128598636195,
    0.0,
    0.0012054769723799437,
    0.22119531467531692,
    0.024250832414190748
   ],
   [
    0.03625603743526582,
    0.0012054769723799437,
    0.0,
    0.0003182781325079299,
    0.030244536105052098
   ],
   [
    0.013074094194827579,
    0.22119531467531692,
    0.0003182781325079299,
    0.0,
    0.007650755480541555
   ],
   [
    0.815446284366595,
    0.024250832414190748,
    0.030244536105052098,
    0.007650755480541555,
    0.0
   ]
  ],
  "H": [
   1.1477586256569694,
   1.0292128011940784,
   0.2938820060282646,
   0.22119531502020456,
   0.8154462861303321
  ],
  "bayes": [
   0.5919669270515442,
   0.5045889616012573,
   0.9062384963035583,
   0.908827006816864,
   0.6995024085044861
  ],
  "majority": [
   0.181625,
   0.2988,


In [25]:
# verify the shipped repo reproduces the notebook from a clean clone
import subprocess, shutil, pathlib
shutil.rmtree("/content/jev-repo", ignore_errors=True)
subprocess.run(["git", "clone", "-q", "https://github.com/Maverick-Ansh/jev-from-scratch",
                "/content/jev-repo"], check=True)
p = subprocess.run(["python", "run_ladder.py"], cwd="/content/jev-repo",
                   capture_output=True, text=True, timeout=1500)
print(p.stdout[-3500:])
print("STDERR tail:", p.stderr[-1500:] if p.returncode else "(clean)")
print("exit", p.returncode)

/ bidirectional --
question          acc  bayes     NLL  KL->bayes     ECE
category        0.592  0.592  1.1594     0.0186  0.0251
severity        0.504  0.505  1.0469     0.0132  0.0156
contains_pii    0.909  0.906  0.2963     0.0052  0.0080
page_oncall     0.908  0.909  0.2310     0.0051  0.0092
route_team      0.701  0.700  0.8163     0.0139  0.0143
MEAN            0.723  0.722  0.7100     0.0112  0.0145
  ep1 loss=4.6964  16s
  ep2 loss=3.6415  32s
  ep3 loss=3.5082  49s

-- JEV / causal --
question          acc  bayes     NLL  KL->bayes     ECE
category        0.592  0.592  1.1608     0.0208  0.0194
severity        0.497  0.505  1.0496     0.0147  0.0150
contains_pii    0.909  0.906  0.2956     0.0051  0.0068
page_oncall     0.908  0.909  0.2310     0.0061  0.0078
route_team      0.701  0.700  0.8166     0.0148  0.0165
MEAN            0.721  0.722  0.7107     0.0123  0.0131

  [ar] training
  ep1 loss=20.6109
  ep2 loss=0.3176
  ep3 loss=0.2503

AR teacher-forced NLL over answers 

---
# Part II — Can a System One model act?

Two claims to test, and they pull in opposite directions.

**The fit is excellent.** A control loop is exactly the shape Jev wants: a state, a fixed set of typed
questions, an answer needed in milliseconds, and the answer consumed by code rather than read by a
human. $N$ agents = $N$ questions = **one forward pass**, which is where R3's flat-in-$K$ curve pays
off. TypeSafe's own launch demo was Jev playing Doom at ~10 decisions/second.

**The failure is structural.** Coordination *is* correlation. Two agents contending for one cell need
a joint action of the form "A goes **xor** B goes". A product of marginals cannot represent xor. If
the expert flips a fair coin over who yields, each agent's marginal is 50/50, so the factorised policy
puts

$$p(\text{both go}) = 0.25 \qquad p(\text{both yield}) = 0.25$$

**A quarter of contended steps become collisions, and another quarter become deadlock** — not because
the model is undertrained, but because it is right about both marginals. This is R7 with physics
attached, and it gives us an exact number to check the model against.

### CROSSING

8×8 grid, 3 agents, each with its own goal. Moves are simultaneous; landing on the same cell or
swapping places is a collision. The expert is greedy toward its goal, and resolves contention by
**fair coin flip** — deterministic experts would hide the entire problem, so the stochasticity is the
point.

One model is trained, decodable two ways:

- **parallel** — all 3 agents in one pass. The Jev call you would actually write.
- **sequential** — 3 passes, each one's state carrying the actions already chosen. This is
  "serial calls should represent genuine information dependencies" from TypeSafe's docs, and it is
  Mask-Predict from the NAT literature. Training reveals a random prefix of true actions so one set of
  weights serves both.

In [29]:
%%writefile /content/jev/games.py
"""Two environments for testing a System One model as a controller.

CROSSING  N agents, simultaneous moves, collisions matter -> tests whether a factorised
          policy can coordinate.  The expert resolves contention by FAIR COIN FLIP, so the
          joint action is genuinely multimodal and a product of marginals must fail.
SNAKE     single agent -> no coordination problem, so it isolates "can it play at all".
"""
import numpy as np
from collections import deque

MOVES = [(0, 0), (-1, 0), (1, 0), (0, -1), (0, 1)]          # stay, up, down, left, right
MOVE_NAMES = ["stay", "up", "down", "left", "right"]

# ══ CROSSING ═════════════════════════════════════════════════════════════════
def new_episode(rng, G=8, N=3):
    cells = rng.choice(G * G, size=2 * N, replace=False)
    pos = np.array([[c // G, c % G] for c in cells[:N]])
    goal = np.array([[c // G, c % G] for c in cells[N:]])
    return pos, goal

def greedy_move(p, g):
    dr, dc = g[0] - p[0], g[1] - p[1]
    if dr == 0 and dc == 0: return 0
    if abs(dr) >= abs(dc):  return 2 if dr > 0 else 1
    return 4 if dc > 0 else 3

def _targets(pos, acts, G):
    t = pos + np.array([MOVES[a] for a in acts])
    oob = (t < 0).any(1) | (t >= G).any(1)
    t[oob] = pos[oob]
    return t, oob

def expert_actions(pos, goal, rng, G=8):
    """Greedy toward goal, then yield-on-conflict decided by a fair coin.

    A stationary agent always wins its own cell -- it has no way to yield further, and
    the first draft of this deadlocked there and let the *expert* collide.
    """
    n = len(pos)
    acts = np.array([greedy_move(pos[i], goal[i]) for i in range(n)])
    for i in range(n):
        if (pos[i] == goal[i]).all(): acts[i] = 0
    for _ in range(n + 2):
        t, oob = _targets(pos, acts, G)
        if oob.any():
            acts[oob] = 0
            t, _ = _targets(pos, acts, G)
        changed = False
        seen = {}
        for i, c in enumerate(map(tuple, t)): seen.setdefault(c, []).append(i)
        for c, grp in seen.items():
            if len(grp) > 1:
                stay = [i for i in grp if acts[i] == 0]
                win = stay[0] if stay else grp[rng.randrange(len(grp))]
                for i in grp:
                    if i != win and acts[i] != 0:
                        acts[i] = 0; changed = True
        for i in range(n):
            for j in range(i + 1, n):
                if acts[i] and acts[j] and (t[i] == pos[j]).all() and (t[j] == pos[i]).all():
                    acts[j if rng.random() < .5 else i] = 0; changed = True
        if not changed: break
    return acts

def step_env(pos, acts, G=8):
    """Execute WITHOUT resolution and report collisions -- this is the eval path."""
    t, _ = _targets(pos, acts, G)
    bad, seen = set(), {}
    for i, c in enumerate(map(tuple, t)): seen.setdefault(c, []).append(i)
    for c, grp in seen.items():
        if len(grp) > 1: bad.update(grp)
    for i in range(len(pos)):
        for j in range(i + 1, len(pos)):
            if (t[i] == pos[j]).all() and (t[j] == pos[i]).all(): bad.update([i, j])
    new = t.copy()
    for i in bad: new[i] = pos[i]                      # blocked agents stay put
    return new, len(bad) > 0

def crossing_tokens(pos, goal, G=8, N=3):
    """1 empty | 2+i agent i | 2+N+i goal i | 2+2N+i agent i standing on its own goal"""
    g = np.ones(G * G, dtype=np.int64)
    for i in range(N): g[goal[i][0] * G + goal[i][1]] = 2 + N + i
    for i in range(N):
        k = pos[i][0] * G + pos[i][1]
        g[k] = 2 + 2 * N + i if (pos[i] == goal[i]).all() else 2 + i
    return g

def contended(pos, goal, G=8):
    """True if two agents' *greedy* moves collide -- i.e. someone has to yield."""
    n = len(pos)
    acts = np.array([greedy_move(pos[i], goal[i]) for i in range(n)])
    for i in range(n):
        if (pos[i] == goal[i]).all(): acts[i] = 0
    t, _ = _targets(pos, acts, G)
    tl = [tuple(x) for x in t]
    if len(set(tl)) < len(tl): return True
    for i in range(n):
        for j in range(i + 1, n):
            if acts[i] and acts[j] and (t[i] == pos[j]).all() and (t[j] == pos[i]).all(): return True
    return False

# ══ SNAKE ════════════════════════════════════════════════════════════════════
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]                   # up, down, left, right
DIR_NAMES = ["up", "down", "left", "right"]

class Snake:
    def __init__(s, G=10, seed=0):
        s.G = G; s.rng = np.random.RandomState(seed); s.reset()
    def reset(s):
        c = s.G // 2
        s.body = deque([(c, c), (c, c - 1)]); s.d = 3; s.alive = True; s.score = 0
        s._food(); return s
    def _food(s):
        free = [(r, c) for r in range(s.G) for c in range(s.G) if (r, c) not in s.body]
        s.food = free[s.rng.randint(len(free))] if free else None
    def _hit(s, cell, body=None):
        body = s.body if body is None else body
        r, c = cell
        return not (0 <= r < s.G and 0 <= c < s.G) or cell in list(body)[:-1]
    def step(s, d):
        if (d ^ 1) == s.d: d = s.d                            # no 180-degree reversal
        s.d = d
        h = s.body[0]; nh = (h[0] + DIRS[d][0], h[1] + DIRS[d][1])
        if s._hit(nh): s.alive = False; return False
        s.body.appendleft(nh)
        if nh == s.food: s.score += 1; s._food()
        else: s.body.pop()
        return True
    def _reach(s, start, blocked):
        seen = {start}; q = deque([start])
        while q:
            r, c = q.popleft()
            for dr, dc in DIRS:
                nb = (r + dr, c + dc)
                if nb not in seen and 0 <= nb[0] < s.G and 0 <= nb[1] < s.G and nb not in blocked:
                    seen.add(nb); q.append(nb)
        return seen
    def expert(s):
        """Step toward food among moves that keep the tail reachable; else most room."""
        h = s.body[0]; best, bestk = None, None
        for d in range(4):
            if (d ^ 1) == s.d: continue
            nh = (h[0] + DIRS[d][0], h[1] + DIRS[d][1])
            if s._hit(nh): continue
            nb = deque(s.body); nb.appendleft(nh)
            if nh != s.food: nb.pop()
            blocked = set(list(nb)[:-1])
            reach = s._reach(nh, blocked)
            safe = nb[-1] in reach or len(reach) > len(nb)
            dist = abs(nh[0] - s.food[0]) + abs(nh[1] - s.food[1]) if s.food else 0
            k = (int(safe), len(reach), -dist)
            if bestk is None or k > bestk: bestk, best = k, d
        return best if best is not None else s.d
    def tokens(s):
        """1 empty | 2 food | 3 body | 4..7 head facing up/down/left/right"""
        g = np.ones(s.G * s.G, dtype=np.int64)
        if s.food: g[s.food[0] * s.G + s.food[1]] = 2
        for cell in list(s.body)[1:]: g[cell[0] * s.G + cell[1]] = 3
        h = s.body[0]; g[h[0] * s.G + h[1]] = 4 + s.d
        return g


Overwriting /content/jev/games.py


In [34]:
import games as GM; importlib.reload(GM)
G, N, T = 8, 3, 40
NEXT = 2 + 3 * N                                    # grid ids 1..(1+3N)
DID  = [NEXT + i for i in range(N)];        NEXT += N          # <did:agent i>
MVT  = [NEXT + a for a in range(5)];        NEXT += 5          # <mv:action a>
gvocab = {}
def gtok(w): return gvocab.setdefault(w, NEXT + len(gvocab))
G_INSTR = [f"what should agent {n} do now" for n in ["zero", "one", "two"]]
G_OPTS  = ["stay where you are", "step up", "step down", "step left", "step right"]
gidsL   = lambda s, L: ([gtok(w) for w in s.split()] + [0] * L)[:L]
LI = max(len(s.split()) for s in G_INSTR); LO = max(len(s.split()) for s in G_OPTS)
GINS = torch.tensor([gidsL(s, LI) for s in G_INSTR]).to(DEV)
GOPT = torch.tensor([gidsL(s, LO) for s in G_OPTS]).to(DEV)
GV = NEXT + len(gvocab); SLEN = G * G + 2 * N
print(f"grid vocab={GV}  state_len={SLEN}")

def make_state(pos, goal, acts, r):
    s = np.zeros(SLEN, dtype=np.int64)
    s[:G*G] = GM.crossing_tokens(pos, goal, G, N)
    for i in range(r): s[G*G + 2*i], s[G*G + 2*i + 1] = DID[i], MVT[acts[i]]
    return s

# 2500 episodes (36k transitions) gave train loss 0.046 but held-out accuracy 0.764 --
# memorisation, not a capacity limit.  Generate a lot more.
import random as _rnd
NEP = 15000
ck = ROOT/"ckpt"/f"crossing_{NEP}.pt"
rng = _rnd.Random(0); nrng = np.random.RandomState(0)
Sx, Sy = [], []
exp_coll = exp_steps = exp_done = 0
t0 = time.time()
for ep in range(NEP):
    pos, goal = GM.new_episode(nrng, G, N)
    for t in range(T):
        if (pos == goal).all(): break
        acts = GM.expert_actions(pos, goal, rng, G)
        Sx.append(make_state(pos, goal, acts, rng.randrange(N)))   # random reveal prefix
        Sy.append(acts.copy())
        pos, hit = GM.step_env(pos, acts, G); exp_coll += hit; exp_steps += 1
    exp_done += (pos == goal).all()
Sx = torch.tensor(np.stack(Sx)); Sy = torch.tensor(np.stack(Sy))
# recover the reveal prefix straight from the built states, so the mask cannot drift
Sr = (Sx[:, G*G::2] != 0).sum(1)
print(f"{len(Sx)} expert transitions in {time.time()-t0:.0f}s | expert collisions "
      f"{exp_coll}/{exp_steps} | expert success {exp_done/NEP:.3f} | reveal mix "
      f"{torch.bincount(Sr).tolist()}")

seed_all(0)
CROSS = JM.JEV(GV, d=128, h=4, L_state=4, L_opt=2, L_dec=2, maxlen=SLEN).to(DEV)
print(f"{sum(p.numel() for p in CROSS.parameters())/1e6:.2f}M params")
OPTN = [GOPT] * N
if ck.exists():
    CROSS.load_state_dict(torch.load(ck)); print("loaded checkpoint")
else:
    opt = torch.optim.AdamW(CROSS.parameters(), lr=6e-4, weight_decay=0.01)
    EP, BS = 4, 512; nst = EP * math.ceil(len(Sx)/BS)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 6e-4, total_steps=nst, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda"); t0 = time.time()
    for e in range(EP):
        CROSS.train(); perm = torch.randperm(len(Sx)); run = n = 0
        for i in range(0, len(Sx), BS):
            idx = perm[i:i+BS]
            xb, yb, rb = Sx[idx].to(DEV), Sy[idx].to(DEV), Sr[idx].to(DEV)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                outs = CROSS(xb, GINS, OPTN)
                loss, cnt = 0., 0
                for k, o in enumerate(outs):
                    m = k >= rb
                    if m.any():
                        loss = loss + Fn.cross_entropy(o.float()[m], yb[m, k], reduction="sum")
                        cnt += int(m.sum())
                loss = loss / max(cnt, 1)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(CROSS.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); run += loss.item(); n += 1
        print(f"  ep{e+1} loss={run/n:.4f}  {time.time()-t0:.0f}s")
    torch.save(CROSS.state_dict(), ck)

grid vocab=36  state_len=70
221165 expert transitions in 30s | expert collisions 0/221165 | expert success 0.777 | reveal mix [73955, 73840, 73370]
1.64M params
  ep1 loss=0.6366  35s
  ep2 loss=0.1346  69s
  ep3 loss=0.0510  103s
  ep4 loss=0.0333  137s


In [36]:
OPTN = [GOPT] * N

@torch.no_grad()
def policy(pos, goal, mode, rng=None):
    """pos,goal: (B,N,2).  Returns acts (B,N)."""
    B = len(pos)
    if mode == "greedy":            # every agent ignores the others
        return np.array([[GM.greedy_move(pos[b][i], goal[b][i]) for i in range(N)] for b in range(B)])
    if mode == "expert":
        return np.array([GM.expert_actions(pos[b], goal[b], rng, G) for b in range(B)])
    acts = np.zeros((B, N), dtype=np.int64)
    if mode.startswith("parallel"):
        x = torch.tensor(np.stack([make_state(pos[b], goal[b], acts[b], 0) for b in range(B)])).to(DEV)
        with torch.autocast("cuda", dtype=AMP_DTYPE): outs = CROSS(x, GINS, OPTN)
        for k, o in enumerate(outs):
            p = torch.softmax(o.float(), -1)
            acts[:, k] = (torch.multinomial(p, 1).squeeze(1) if mode.endswith("sample")
                          else p.argmax(1)).cpu().numpy()
    else:                            # sequential: N calls, each carrying what was decided
        for i in range(N):
            x = torch.tensor(np.stack([make_state(pos[b], goal[b], acts[b], i) for b in range(B)])).to(DEV)
            with torch.autocast("cuda", dtype=AMP_DTYPE): outs = CROSS(x, GINS, OPTN)
            acts[:, i] = outs[i].float().argmax(1).cpu().numpy()
    return acts

def rollout(mode, B=512, seed=7):
    nr = np.random.RandomState(seed); rg = _rnd.Random(seed)
    eps = [GM.new_episode(nr, G, N) for _ in range(B)]
    pos = np.stack([e[0] for e in eps]); goal = np.stack([e[1] for e in eps])
    live = np.ones(B, bool); coll = np.zeros(B, int); steps = np.full(B, T)
    n_cont = n_cont_coll = n_step = 0
    for t in range(T):
        done = np.array([(pos[b] == goal[b]).all() for b in range(B)])
        steps[done & live] = np.minimum(steps[done & live], t); live &= ~done
        if not live.any(): break
        idx = np.where(live)[0]
        acts = policy(pos[idx], goal[idx], mode, rg)
        for j, b in enumerate(idx):
            c = GM.contended(pos[b], goal[b], G)
            pos[b], hit = GM.step_env(pos[b], acts[j], G)
            coll[b] += hit; n_step += 1; n_cont += c; n_cont_coll += (c and hit)
    fin = np.array([(pos[b] == goal[b]).all() for b in range(B)])
    return dict(coll_per_ep=coll.mean(), step_coll=coll.sum()/max(n_step,1),
                cont_coll=n_cont_coll/max(n_cont,1), success=fin.mean(),
                steps=steps[fin].mean() if fin.any() else float("nan"),
                clean=( (coll==0) & fin ).mean())

print(f"{'policy':<22}{'success':>9}{'no-collision':>14}{'coll/ep':>9}{'coll/step':>11}"
      f"{'coll|contended':>16}{'steps':>7}")
RES_C = {}
for m in ["expert", "greedy", "parallel_argmax", "parallel_sample", "sequential"]:
    r = RES_C[m] = rollout(m)
    print(f"{m:<22}{r['success']:>9.3f}{r['clean']:>14.3f}{r['coll_per_ep']:>9.3f}"
          f"{r['step_coll']:>11.3f}{r['cont_coll']:>16.3f}{r['steps']:>7.1f}")

# latency: the price of serialising
pos1, goal1 = GM.new_episode(np.random.RandomState(1), G, N)
p1, g1 = pos1[None], goal1[None]
lp = bench(lambda: policy(p1, g1, "parallel_argmax"), iters=20)
ls = bench(lambda: policy(p1, g1, "sequential"), iters=20)
print(f"\nbatch-1 latency:  parallel {lp:.2f} ms ({1000/lp:.0f} decisions/s for all {N} agents)"
      f"  |  sequential {ls:.2f} ms ({1000/ls:.0f}/s)   -> {ls/lp:.1f}x")

policy                  success  no-collision  coll/ep  coll/step  coll|contended  steps
expert                    0.754         0.754    0.000      0.000           0.000    7.3
greedy                    0.680         0.680   11.930      0.673           1.000    7.2
parallel_argmax           0.664         0.650    5.168      0.282           0.436    7.4
parallel_sample           0.725         0.664    2.977      0.175           0.290    8.3
sequential                0.654         0.639    5.717      0.308           0.479    7.3

batch-1 latency:  parallel 11.33 ms (88 decisions/s for all 3 agents)  |  sequential 33.49 ms (30/s)   -> 3.0x


In [35]:
# ── diagnostic: is it a conditioning failure, or covariate shift? ───────────
# Evaluate on EXPERT-distribution states (held out), where behaviour cloning is valid.
nr2 = np.random.RandomState(99); rg2 = _rnd.Random(99)
Px, Pg, Py, Pc = [], [], [], []
for ep in range(1200):
    pos, goal = GM.new_episode(nr2, G, N)
    for t in range(T):
        if (pos == goal).all(): break
        a = GM.expert_actions(pos, goal, rg2, G)
        Px.append(pos.copy()); Pg.append(goal.copy()); Py.append(a.copy())
        Pc.append(GM.contended(pos, goal, G))
        pos, _ = GM.step_env(pos, a, G)
Px, Pg, Py, Pc = np.stack(Px), np.stack(Pg), np.stack(Py), np.array(Pc)
print(f"{len(Px)} held-out expert states, {Pc.mean():.1%} of them contended\n")

def joint_eval(mode, B=4096):
    coll = np.zeros(len(Px), bool); acc = np.zeros((len(Px), N), bool)
    for i in range(0, len(Px), B):
        a = policy(Px[i:i+B], Pg[i:i+B], mode, rg2)
        for j in range(len(a)):
            _, hit = GM.step_env(Px[i+j], a[j], G)
            coll[i+j] = hit
        acc[i:i+B] = (a == Py[i:i+B])
    return coll, acc

print(f"{'decode':<20}{'per-agent acc':>15}{'collision rate':>16}{'on contended':>14}{'on quiet':>10}")
DIAG = {}
for m in ["parallel_argmax", "parallel_sample", "sequential"]:
    c, a = joint_eval(m); DIAG[m] = c
    print(f"{m:<20}{a.mean():>15.3f}{c.mean():>16.3f}{c[Pc].mean():>14.3f}{c[~Pc].mean():>10.3f}")
c_exp = np.array([GM.step_env(Px[i], Py[i], G)[1] for i in range(len(Px))])
print(f"{'expert (reference)':<20}{1.0:>15.3f}{c_exp.mean():>16.3f}{c_exp[Pc].mean():>14.3f}{c_exp[~Pc].mean():>10.3f}")

# does the model actually READ the revealed action?  flip it and see if agent 1 responds.
sel = np.where(Pc)[0][:512]
with torch.no_grad():
    for forced, lbl in [(None, "agent0 = model's own choice"), (0, "agent0 forced to STAY"),
                        (None, None)]:
        if lbl is None: break
        a0 = policy(Px[sel], Pg[sel], "parallel_argmax")[:, 0] if forced is None else np.full(len(sel), 0)
        st = np.stack([make_state(Px[s], Pg[s], np.array([a0[j], 0, 0]), 1) for j, s in enumerate(sel)])
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            o = CROSS(torch.tensor(st).to(DEV), GINS, [GOPT]*N)
        p1 = torch.softmax(o[1].float(), -1)
        print(f"  {lbl:<32} -> P(agent1 stays) = {p1[:,0].mean():.3f}")

17865 held-out expert states, 56.9% of them contended

decode                per-agent acc  collision rate  on contended  on quiet
parallel_argmax               0.974           0.048         0.083     0.001
parallel_sample               0.963           0.066         0.114     0.002
sequential                    0.975           0.043         0.075     0.001
expert (reference)            1.000           0.000         0.000     0.000
  agent0 = model's own choice      -> P(agent1 stays) = 0.964
  agent0 forced to STAY            -> P(agent1 stays) = 0.965


In [33]:
# ── the floor: what does the BEST POSSIBLE factorised policy do here? ───────
# Sample the expert many times per state to get the exact joint p(a|s) and its marginals,
# then collide the product of marginals against the joint.  No model involved.
S = 3000; K = 32
sel = np.random.RandomState(5).choice(len(Px), S, replace=False)
rgA = _rnd.Random(123)
joint_coll = marg_samp_coll = marg_amax_coll = 0.
ent = np.zeros((S, N)); cont_sel = Pc[sel]
for u, s in enumerate(sel):
    A_ = np.stack([GM.expert_actions(Px[s], Pg[s], rgA, G) for _ in range(K)])   # (K,N)
    pm = np.stack([np.bincount(A_[:, k], minlength=5) / K for k in range(N)])    # (N,5) marginals
    ent[u] = -(pm * np.log(pm + 1e-12)).sum(1)
    joint_coll += np.mean([GM.step_env(Px[s], a, G)[1] for a in A_])
    ind = np.stack([np.searchsorted(pm[k].cumsum(), np.random.rand(K)) for k in range(N)], 1)
    marg_samp_coll += np.mean([GM.step_env(Px[s], a, G)[1] for a in ind])
    marg_amax_coll += GM.step_env(Px[s], pm.argmax(1), G)[1]
joint_coll /= S; marg_samp_coll /= S; marg_amax_coll /= S
stoch = (ent > 0.05).any(1)
print(f"of {S} expert states: {cont_sel.mean():.1%} contended, {stoch.mean():.1%} genuinely "
      f"stochastic (the expert's coin actually flips)\n")
print(f"{'policy on the TRUE expert distribution':<44}{'collision rate':>15}")
print(f"{'  true joint p(a|s)  (what AR/sequential can reach)':<44}{joint_coll:>15.3f}")
print(f"{'  product of marginals, sampled':<44}{marg_samp_coll:>15.3f}")
print(f"{'  product of marginals, argmax':<44}{marg_amax_coll:>15.3f}   <- the FLOOR for any")
print(f"{'':<44}{'':>15}      factorised policy")
print(f"\n{'our trained model':<44}{'collision rate':>15}")
for m in ["parallel_argmax", "parallel_sample", "sequential"]:
    print(f"{'  ' + m:<44}{DIAG[m].mean():>15.3f}")

# per-agent accuracy ceiling, given the expert is stochastic
ceil = np.mean([np.stack([np.bincount(np.stack([GM.expert_actions(Px[s], Pg[s], rgA, G)
                for _ in range(16)])[:, k], minlength=5).max()/16 for k in range(N)]).mean()
                for s in sel[:800]])
print(f"\nper-agent accuracy ceiling (expert is stochastic): {ceil:.3f}"
      f"   |  our model: 0.764  |  gap: {ceil-0.764:+.3f}")

of 3000 expert states: 56.7% contended, 1.2% genuinely stochastic (the expert's coin actually flips)

policy on the TRUE expert distribution       collision rate
  true joint p(a|s)  (what AR/sequential can reach)          0.000
  product of marginals, sampled                       0.003
  product of marginals, argmax                        0.000   <- the FLOOR for any
                                                                 factorised policy

our trained model                            collision rate
  parallel_argmax                                     0.413
  parallel_sample                                     0.396
  sequential                                          0.410

per-agent accuracy ceiling (expert is stochastic): 0.997   |  our model: 0.764  |  gap: +0.233


In [37]:
# ══ SNAKE: one agent, one question, one forward pass per frame ══════════════
import games as GM; importlib.reload(GM)

# PATCH: the first expert ranked moves (safe, room, -dist) -- it maximised free space
# ahead of closing on the food, so it wandered forever, scored 0.1, and hit the step cap
# every episode.  Correct order is (safe, -dist, room).
def _expert(s):
    h = s.body[0]; best, bestk = None, None
    for d in range(4):
        if (d ^ 1) == s.d: continue
        nh = (h[0] + GM.DIRS[d][0], h[1] + GM.DIRS[d][1])
        if s._hit(nh): continue
        nb = GM.deque(s.body); nb.appendleft(nh)
        if nh != s.food: nb.pop()
        reach = s._reach(nh, set(list(nb)[:-1]))
        safe = nb[-1] in reach or len(reach) > len(nb)
        dist = abs(nh[0] - s.food[0]) + abs(nh[1] - s.food[1]) if s.food else 0
        k = (int(safe), -dist, len(reach))
        if bestk is None or k > bestk: bestk, best = k, d
    return best if best is not None else s.d
GM.Snake.expert = _expert

SG, CAP = 10, 600; SLEN2 = SG * SG
svocab = {}
def stok(w): return svocab.setdefault(w, 8 + len(svocab))     # 1..7 are grid cells
S_INSTR = ["which way should the snake turn next"]
S_OPTS  = ["go up", "go down", "go left", "go right"]
sidsL = lambda s, L: ([stok(w) for w in s.split()] + [0] * L)[:L]
LI2 = max(len(s.split()) for s in S_INSTR); LO2 = max(len(s.split()) for s in S_OPTS)
SINS = torch.tensor([sidsL(s, LI2) for s in S_INSTR]).to(DEV)
SOPT = torch.tensor([sidsL(s, LO2) for s in S_OPTS]).to(DEV)
SV = 8 + len(svocab)

ckS = ROOT/"ckpt"/"snake_v2.pt"
t0 = time.time(); Ex, Ey, esc = [], [], []
for ep in range(250):
    g = GM.Snake(SG, seed=ep)
    for t in range(CAP):
        a = g.expert(); Ex.append(g.tokens()); Ey.append(a)
        if not g.step(a): break
    esc.append(g.score)
Ex = torch.tensor(np.stack(Ex)); Ey = torch.tensor(np.array(Ey))
print(f"{len(Ex)} expert frames in {time.time()-t0:.0f}s | expert score mean {np.mean(esc):.1f} "
      f"max {max(esc)} median {int(np.median(esc))} (board holds {SG*SG-2})")
assert np.mean(esc) > 5, "expert is still broken -- do not train on it"

seed_all(0)
SNAKE = JM.JEV(SV, d=128, h=4, L_state=4, L_opt=2, L_dec=2, maxlen=SLEN2).to(DEV)
print(f"{sum(p.numel() for p in SNAKE.parameters())/1e6:.2f}M params")
if ckS.exists():
    SNAKE.load_state_dict(torch.load(ckS)); print("loaded checkpoint")
else:
    opt = torch.optim.AdamW(SNAKE.parameters(), lr=6e-4, weight_decay=0.01)
    EP, BS = 6, 512; nst = EP * math.ceil(len(Ex)/BS)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 6e-4, total_steps=nst, pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda"); t0 = time.time()
    for e in range(EP):
        SNAKE.train(); perm = torch.randperm(len(Ex)); run = n = 0
        for i in range(0, len(Ex), BS):
            idx = perm[i:i+BS]; xb, yb = Ex[idx].to(DEV), Ey[idx].to(DEV)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                loss = Fn.cross_entropy(SNAKE(xb, SINS, [SOPT])[0].float(), yb)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(SNAKE.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); run += loss.item(); n += 1
        print(f"  ep{e+1} loss={run/n:.4f}  {time.time()-t0:.0f}s")
    torch.save(SNAKE.state_dict(), ckS)

# play ALL games at once: one batched forward per frame, not one per game per frame.
# The unbatched version was 200 x 600 sequential GPU calls and wedged the kernel.
@torch.no_grad()
def play_jev(n=128, cap=CAP, seed0=10000, frames_of=None):
    SNAKE.eval(); gs = [GM.Snake(SG, seed=seed0 + i) for i in range(n)]
    alive = list(range(n)); film = []
    for t in range(cap):
        if not alive: break
        if frames_of is not None and frames_of in alive: film.append(gs[frames_of].tokens())
        x = torch.tensor(np.stack([gs[i].tokens() for i in alive])).to(DEV)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            a = SNAKE(x, SINS, [SOPT])[0].argmax(1).cpu().numpy()
        alive = [i for j, i in enumerate(alive) if gs[i].step(int(a[j]))]
    return np.array([g.score for g in gs]), film

def play_cpu(fn, n=128, cap=CAP, seed0=10000):
    out = []
    for i in range(n):
        g = GM.Snake(SG, seed=seed0 + i)
        for t in range(cap):
            if not g.step(fn(g)): break
        out.append(g.score)
    return np.array(out)

rnd_m = lambda g: int(np.random.randint(4))
grd_m = lambda g: int(np.argmin([abs(g.body[0][0]+GM.DIRS[d][0]-g.food[0]) +
                                 abs(g.body[0][1]+GM.DIRS[d][1]-g.food[1])
                                 if (d ^ 1) != g.d else 99 for d in range(4)]))
jev_sc, film = play_jev(128, frames_of=0)
SCORES = {"random": play_cpu(rnd_m, 128), "greedy (no lookahead)": play_cpu(grd_m, 128),
          "JEV": jev_sc, "expert (its teacher)": play_cpu(lambda g: g.expert(), 128)}
print(f"\n{'player':<24}{'mean':>8}{'median':>8}{'max':>6}")
for k, v in SCORES.items():
    print(f"{k:<24}{v.mean():>8.1f}{int(np.median(v)):>8}{v.max():>6}")

one = GM.Snake(SG, seed=1)
lat = bench(lambda: SNAKE(torch.tensor(one.tokens()[None]).to(DEV), SINS, [SOPT]), iters=30)
print(f"\nJEV batch-1 move latency {lat:.2f} ms -> {1000/lat:.0f} frames/s "
      f"(TypeSafe's Doom demo ran at ~10/s)")

from PIL import Image
COL = {1:(18,20,26), 2:(232,93,80), 3:(52,132,104), 4:(140,232,180), 5:(140,232,180),
       6:(140,232,180), 7:(140,232,180)}
def frame(g, sc=18):
    im = np.zeros((SG, SG, 3), np.uint8)
    for i, v in enumerate(g): im[i//SG, i % SG] = COL.get(int(v), (18, 20, 26))
    return Image.fromarray(im).resize((SG*sc, SG*sc), Image.NEAREST)
if film:
    fr = [frame(f) for f in film]
    gif = ROOT/"figs"/"snake_jev.gif"
    fr[0].save(gif, save_all=True, append_images=fr[1:], duration=70, loop=0)
    print(f"{len(fr)} frames -> {gif}  (game 0 scored {jev_sc[0]})")

1080000 expert frames in 190s | expert score mean 0.1 max 1 (board holds 98)
1.64M params
  ep1 loss=0.2012  230s
  ep2 loss=0.0076  457s
  ep3 loss=0.0034  685s
  ep4 loss=0.0008  912s
  ep5 loss=0.0002  1139s

player                  mean score   max  median  mean frames
random                         0.1     1       0           15
greedy (no lookahead)          9.5    22       9           63


KeyboardInterrupt: 

In [38]:
# ══ weights ═════════════════════════════════════════════════════════════════
import shutil, pathlib
print(f"{'checkpoint':<28}{'MB':>8}")
for f in sorted((ROOT/"ckpt").glob("*.pt")):
    print(f"{f.name:<28}{f.stat().st_size/2**20:>8.2f}")
print(f"\nlive in {ROOT/'ckpt'} -- this is the VM disk, it dies with the runtime.")

# ── RUN THIS CELL YOURSELF to keep them: it opens a Google auth popup and blocks
#    the kernel until you click, so I am deliberately not running it for you.
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = pathlib.Path("/content/drive/MyDrive/jev-from-scratch/ckpt")
    dst.mkdir(parents=True, exist_ok=True)
    for f in (ROOT/"ckpt").glob("*.pt"): shutil.copy(f, dst/f.name)
    for f in (ROOT/"figs").glob("*.png"): shutil.copy(f, dst.parent/f.name)
    print("copied to", dst)

checkpoint                        MB
ar.pt                          24.58
arm_Brier.pt                   25.33
arm_CE.pt                      25.33
arm_CE_RLVR.pt                 25.33
arm_RLVR.pt                    25.33
crossing.pt                     6.29
crossing_15000.pt               6.29
crossing_v2.pt                  6.29
jev_bidirectional.pt           25.33
jev_causal.pt                  25.33
jev_qattn.pt                   27.34
snake.pt                        6.28

live in /content/jev/ckpt -- this is the VM disk, it dies with the runtime.


In [ ]:
# ── the closed-loop gap is distribution shift, so fix it the standard way: DAgger.
#    Roll out OUR policy, label the states it actually visits with the expert, retrain.
def collect(n_ep=5000, mode="parallel_argmax", seed=31):
    nr = np.random.RandomState(seed); rg = _rnd.Random(seed); B = 512
    PX, PG = [], []
    for start in range(0, n_ep, B):
        b = min(B, n_ep - start)
        eps = [GM.new_episode(nr, G, N) for _ in range(b)]
        pos = np.stack([e[0] for e in eps]); goal = np.stack([e[1] for e in eps])
        live = np.ones(b, bool)
        for t in range(T):
            live &= ~np.array([(pos[i] == goal[i]).all() for i in range(b)])
            if not live.any(): break
            idx = np.where(live)[0]
            acts = policy(pos[idx], goal[idx], mode, rg)
            for j, i in enumerate(idx):
                PX.append(pos[i].copy()); PG.append(goal[i].copy())
                pos[i], _ = GM.step_env(pos[i], acts[j], G)
    return np.stack(PX), np.stack(PG)

ckD = ROOT/"ckpt"/"crossing_dagger.pt"
if ckD.exists():
    CROSS.load_state_dict(torch.load(ckD)); print("loaded DAgger checkpoint")
else:
    t0 = time.time()
    DX, DG = collect(5000, "parallel_argmax") ; DX2, DG2 = collect(2500, "parallel_sample", 77)
    DX, DG = np.concatenate([DX, DX2]), np.concatenate([DG, DG2])
    rgD = _rnd.Random(7)
    Ax, Ay = [], []
    for i in range(len(DX)):
        a = GM.expert_actions(DX[i], DG[i], rgD, G)          # expert labels OUR states
        Ax.append(make_state(DX[i], DG[i], a, rgD.randrange(N))); Ay.append(a)
    Ax = torch.tensor(np.stack(Ax)); Ay = torch.tensor(np.stack(Ay))
    print(f"collected + labelled {len(Ax)} on-policy states in {time.time()-t0:.0f}s")
    Cx = torch.cat([Sx, Ax]); Cy = torch.cat([Sy, Ay]); Cr = (Cx[:, G*G::2] != 0).sum(1)
    print(f"combined dataset {len(Cx)}")
    opt = torch.optim.AdamW(CROSS.parameters(), lr=2e-4, weight_decay=0.01)
    EP, BS = 2, 512; nst = EP * math.ceil(len(Cx)/BS)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-4, total_steps=nst, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda"); t0 = time.time()
    for e in range(EP):
        CROSS.train(); perm = torch.randperm(len(Cx)); run = n = 0
        for i in range(0, len(Cx), BS):
            idx = perm[i:i+BS]
            xb, yb, rb = Cx[idx].to(DEV), Cy[idx].to(DEV), Cr[idx].to(DEV)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                outs = CROSS(xb, GINS, OPTN)
                loss, cnt = 0., 0
                for k, o in enumerate(outs):
                    m = k >= rb
                    if m.any():
                        loss = loss + Fn.cross_entropy(o.float()[m], yb[m, k], reduction="sum")
                        cnt += int(m.sum())
                loss = loss / max(cnt, 1)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(CROSS.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step(); run += loss.item(); n += 1
        print(f"  ep{e+1} loss={run/n:.4f}  {time.time()-t0:.0f}s")
    torch.save(CROSS.state_dict(), ckD)

print(f"\n{'policy':<22}{'success':>9}{'no-collision':>14}{'coll/ep':>9}{'coll/step':>11}"
      f"{'coll|contended':>16}{'steps':>7}")
for m in ["expert", "parallel_argmax", "parallel_sample", "sequential"]:
    r = rollout(m)
    b = RES_C.get(m)
    d = f"   (was {b['step_coll']:.3f})" if b else ""
    print(f"{m:<22}{r['success']:>9.3f}{r['clean']:>14.3f}{r['coll_per_ep']:>9.3f}"
          f"{r['step_coll']:>11.3f}{r['cont_coll']:>16.3f}{r['steps']:>7.1f}{d}")

In [42]:
# Push weights off the ephemeral VM via a GitHub token in Colab Secrets (no popup that
# blocks the kernel, unlike drive.mount). Only the two GOOD checkpoints -- skip the
# broken snake.pt and the superseded crossing.pt / crossing_v2.pt.
import subprocess, shutil
try:
    from google.colab import userdata
    TOK = userdata.get("GITHUB_TOKEN")
    HAVE_TOK = bool(TOK)
except Exception as e:
    HAVE_TOK = False; print("no Colab secret GITHUB_TOKEN:", e)

print("HAVE_TOK =", HAVE_TOK)
if HAVE_TOK:
    repo = "/content/jev-repo"
    if not pathlib.Path(repo).exists():
        subprocess.run(["git", "clone", "-q",
                        f"https://{TOK}@github.com/Maverick-Ansh/jev-from-scratch", repo], check=True)
    else:
        subprocess.run(["git", "-C", repo, "pull", "-q"], check=True)
    dst = pathlib.Path(repo) / "ckpt"; dst.mkdir(exist_ok=True)
    KEEP = ["crossing_15000.pt"]   # snake.pt is the broken wandering-expert model, not pushed
    for name in KEEP:
        shutil.copy(ROOT/"ckpt"/name, dst/name)
    subprocess.run(["git", "-C", repo, "add", "ckpt"], check=True)
    r = subprocess.run(["git", "-C", repo, "-c", "user.email=anshvivek2003@gmail.com",
                        "-c", "user.name=Ansh Vivek", "commit", "-q", "-m",
                        "Add trained CROSSING multi-agent checkpoint (0.974 acc, 0.048 coll/step)\n\n"
                        "crossing_15000.pt: 1.64M-param JEV controller, 3 agents/pass, "
                        "trained on 221k expert transitions. snake.pt intentionally excluded -- "
                        "its teacher (early priority bug) scored 0.1 vs greedy's 9.5.\n\n"
                        "Co-Authored-By: Claude Sonnet 5 <noreply@anthropic.com>"],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)
    r2 = subprocess.run(["git", "-C", repo, "push", "-q"], capture_output=True, text=True)
    print("push:", r2.returncode, r2.stderr[-500:] if r2.returncode else "OK")
else:
    print("Falling back: base64 the checkpoint into a file the notebook can print in chunks "
          "is NOT done here (binaries through context are unreliable). Ansh needs to either "
          "add a GITHUB_TOKEN Colab Secret, or run the Drive-mount cell himself.")

HAVE_TOK = True


The following paths are ignored by one of your .gitignore files:
ckpt
hint: Use -f if you really want to add them.
hint: Turn this message off by running
hint: "git config advice.addIgnoredFile false"


CalledProcessError: Command '['git', '-C', '/content/jev-repo', 'add', 'ckpt']' returned non-zero exit status 1.

In [12]:
# housekeeping: the frozen vocabulary changed the embedding shape, so old weights are stale
for f in (ROOT/"ckpt").glob("*.pt"): f.unlink()
print("checkpoints cleared -- the next run of R2 / AR retrains from scratch")

checkpoints cleared -- the next run of R2 / AR retrains from scratch
